# Chapter 9: System

> *"The purpose of abstraction is not to be vague, but to create a new
> semantic level in which one can be absolutely precise."*  
> — **Edsger W. Dijkstra**

------------------------------------------------------------------------

In Part I of this book, we inhabited a pristine mathematical landscape.
Using Python, we explored numbers, logic, pure functions, and abstract
data types. In that world, variables are symbols, memory is infinite and
invisible, and execution occurs via high-level evaluation models. If our
program requires a new data structure, the runtime allocates it; when we
discard it, the garbage collector whisks it away.

Yet real programs do not run on abstract mathematical models; they run
on physical silicon. The microprocessors powering your laptop,
smartphone, and cloud servers have no innate concept of closures,
objects, recursion, or lexical scope. Beneath the compiler lies a
radically simpler, unforgiving machine: a clock ticking billions of
times per second, moving fixed-width bit patterns across microscopic
copper buses between banks of memory and hardwired arithmetic circuits.

In Part II, we shift our medium to **Modern C++**. Where Python lifts
code into higher tiers of abstraction, C++ provides the dual power of
zero-overhead abstractions coupled with direct, unmediated mechanical
sympathy with the physical hardware.

To understand why programming languages behave the way they do, we must
first build an exact mental and programmatic model of the machine
itself. In this chapter, we will strip away the operating system and
runtimes to explore the **Von Neumann architecture**, the
**Fetch-Decode-Execute cycle**, hardware registers, and machine-level
instruction sets. Finally, we will implement a fully functioning,
cycle-accurate virtual computer system in Modern C++.

### 9.1 The Illusion of the High-Level Machine

Consider a simple conditional loop written in C++:

``` cpp
int total = 0, i = 1;

while (i <= 5) {
    total += i * 2;
    i += 1;
}
```

When processed by a C++ compiler, this code undergoes multiple stages of
parsing, abstract syntax tree (AST) construction,
intermediate-representation (IR) translation, and object-code
(machine-code) emission. But what does the central processing unit (CPU)
actually see at runtime?

The CPU possesses no intrinsic concept of a `while` loop, nor does it
know what an identifier named `total` represents. To execute this
snippet, the high-level semantic structures must be dismantled into
primitive operations:

1.  Allocating specific physical storage slots (registers or memory
    words) for `total` and `i`.
2.  Emitting an explicit compare instruction to test if the value in `i`
    exceeds `5`.
3.  Executing a conditional branch instruction that jumps over the loop
    body if the comparison evaluates to `true`.
4.  Performing arithmetic shifts or multiplication on hardware
    registers.
5.  Emitting an unconditional jump instruction back to the loop header.

The translation pipeline can be visualised as a stack of increasingly
concrete abstractions (Figure 9–1):

<figure id="fig:abstraction">
<img src="data:image/svg+xml;base64,PD94bWwgdmVyc2lvbj0iMS4wIiBlbmNvZGluZz0iVVRGLTgiIHN0YW5kYWxvbmU9Im5vIj8+Cjwh
RE9DVFlQRSBzdmcgUFVCTElDICItLy9XM0MvL0RURCBTVkcgMS4xLy9FTiIKICJodHRwOi8vd3d3
LnczLm9yZy9HcmFwaGljcy9TVkcvMS4xL0RURC9zdmcxMS5kdGQiPgo8IS0tIEdlbmVyYXRlZCBi
eSBncmFwaHZpeiB2ZXJzaW9uIDIuNDMuMCAoMCkKIC0tPgo8IS0tIFRpdGxlOiBHIFBhZ2VzOiAx
IC0tPgo8c3ZnIHdpZHRoPSIzMzlwdCIgaGVpZ2h0PSIzOThwdCIKIHZpZXdCb3g9IjAuMDAgMC4w
MCAzMzguODggMzk4LjAwIiB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHhtbG5z
OnhsaW5rPSJodHRwOi8vd3d3LnczLm9yZy8xOTk5L3hsaW5rIj4KPGcgaWQ9ImdyYXBoMCIgY2xh
c3M9ImdyYXBoIiB0cmFuc2Zvcm09InNjYWxlKDEgMSkgcm90YXRlKDApIHRyYW5zbGF0ZSg0IDM5
NCkiPgo8dGl0bGU+RzwvdGl0bGU+CjwhLS0gaGlnaCAtLT4KPGcgaWQ9Im5vZGUxIiBjbGFzcz0i
bm9kZSI+Cjx0aXRsZT5oaWdoPC90aXRsZT4KPGVsbGlwc2UgZmlsbD0ibm9uZSIgc3Ryb2tlPSJi
bGFjayIgY3g9IjE2MS44OCIgY3k9Ii0zNzIiIHJ4PSIxNDAuODgiIHJ5PSIxOCIvPgo8dGV4dCB0
ZXh0LWFuY2hvcj0ibWlkZGxlIiB4PSIxNjEuODgiIHk9Ii0zNjkuNSIgZm9udC1mYW1pbHk9IkNv
dXJpZXIgTmV3LCBEZWphVnUgU2FucyBNb25vLCBtb25vc3BhY2UiIGZvbnQtc2l6ZT0iMTAuMDAi
PkhpZ2gmIzQ1O0xldmVsIFNvdXJjZSBDb2RlIChQeXRob24gLyBDKyspPC90ZXh0Pgo8L2c+Cjwh
LS0gaW50ZXJtZWRpYXRlIC0tPgo8ZyBpZD0ibm9kZTIiIGNsYXNzPSJub2RlIj4KPHRpdGxlPmlu
dGVybWVkaWF0ZTwvdGl0bGU+CjxlbGxpcHNlIGZpbGw9Im5vbmUiIHN0cm9rZT0iYmxhY2siIGN4
PSIxNjEuODgiIGN5PSItMjg5IiByeD0iMTYxLjc2IiByeT0iMTgiLz4KPHRleHQgdGV4dC1hbmNo
b3I9Im1pZGRsZSIgeD0iMTYxLjg4IiB5PSItMjg2LjUiIGZvbnQtZmFtaWx5PSJDb3VyaWVyIE5l
dywgRGVqYVZ1IFNhbnMgTW9ubywgbW9ub3NwYWNlIiBmb250LXNpemU9IjEwLjAwIj5JbnRlcm1l
ZGlhdGUgUmVwcmVzZW50YXRpb24gKElSIC8gQnl0ZWNvZGUpPC90ZXh0Pgo8L2c+CjwhLS0gaGln
aCYjNDU7Jmd0O2ludGVybWVkaWF0ZSAtLT4KPGcgaWQ9ImVkZ2UxIiBjbGFzcz0iZWRnZSI+Cjx0
aXRsZT5oaWdoJiM0NTsmZ3Q7aW50ZXJtZWRpYXRlPC90aXRsZT4KPHBhdGggZmlsbD0ibm9uZSIg
c3Ryb2tlPSJibGFjayIgZD0iTTE2MS44OCwtMzUzLjgyQzE2MS44OCwtMzQzLjE5IDE2MS44OCwt
MzI5LjMxIDE2MS44OCwtMzE3LjIiLz4KPHBvbHlnb24gZmlsbD0iYmxhY2siIHN0cm9rZT0iYmxh
Y2siIHBvaW50cz0iMTY1LjM4LC0zMTcuMTUgMTYxLjg4LC0zMDcuMTUgMTU4LjM4LC0zMTcuMTUg
MTY1LjM4LC0zMTcuMTUiLz4KPHRleHQgdGV4dC1hbmNob3I9Im1pZGRsZSIgeD0iMjM3LjM4IiB5
PSItMzI4IiBmb250LWZhbWlseT0iQ291cmllciBOZXcsIERlamFWdSBTYW5zIE1vbm8sIG1vbm9z
cGFjZSIgZm9udC1zaXplPSIxMC4wMCI+ICYjMTYwOyYjMTYwO0NvbXBpbGF0aW9uIC8gTG93ZXJp
bmc8L3RleHQ+CjwvZz4KPCEtLSBhc3NlbWJseSAtLT4KPGcgaWQ9Im5vZGUzIiBjbGFzcz0ibm9k
ZSI+Cjx0aXRsZT5hc3NlbWJseTwvdGl0bGU+CjxlbGxpcHNlIGZpbGw9Im5vbmUiIHN0cm9rZT0i
YmxhY2siIGN4PSIxNjEuODgiIGN5PSItMTk1IiByeD0iMTIzLjA2IiByeT0iMTgiLz4KPHRleHQg
dGV4dC1hbmNob3I9Im1pZGRsZSIgeD0iMTYxLjg4IiB5PSItMTkyLjUiIGZvbnQtZmFtaWx5PSJD
b3VyaWVyIE5ldywgRGVqYVZ1IFNhbnMgTW9ubywgbW9ub3NwYWNlIiBmb250LXNpemU9IjEwLjAw
Ij5Bc3NlbWJseSBMYW5ndWFnZSAoeDg2JiM0NTs2NCAvIEFSTSk8L3RleHQ+CjwvZz4KPCEtLSBp
bnRlcm1lZGlhdGUmIzQ1OyZndDthc3NlbWJseSAtLT4KPGcgaWQ9ImVkZ2UyIiBjbGFzcz0iZWRn
ZSI+Cjx0aXRsZT5pbnRlcm1lZGlhdGUmIzQ1OyZndDthc3NlbWJseTwvdGl0bGU+CjxwYXRoIGZp
bGw9Im5vbmUiIHN0cm9rZT0iYmxhY2siIGQ9Ik0xNjEuODgsLTI3MC43QzE2MS44OCwtMjU3LjQ2
IDE2MS44OCwtMjM4Ljk1IDE2MS44OCwtMjIzLjY2Ii8+Cjxwb2x5Z29uIGZpbGw9ImJsYWNrIiBz
dHJva2U9ImJsYWNrIiBwb2ludHM9IjE2NS4zOCwtMjIzLjIzIDE2MS44OCwtMjEzLjIzIDE1OC4z
OCwtMjIzLjIzIDE2NS4zOCwtMjIzLjIzIi8+Cjx0ZXh0IHRleHQtYW5jaG9yPSJtaWRkbGUiIHg9
IjI0Ni4zOCIgeT0iLTI0NSIgZm9udC1mYW1pbHk9IkNvdXJpZXIgTmV3LCBEZWphVnUgU2FucyBN
b25vLCBtb25vc3BhY2UiIGZvbnQtc2l6ZT0iMTAuMDAiPiAmIzE2MDsmIzE2MDsmIzE2MDsmIzE2
MDtJbnN0cnVjdGlvbiBTZWxlY3Rpb24gLzwvdGV4dD4KPHRleHQgdGV4dC1hbmNob3I9Im1pZGRs
ZSIgeD0iMjQ2LjM4IiB5PSItMjM0IiBmb250LWZhbWlseT0iQ291cmllciBOZXcsIERlamFWdSBT
YW5zIE1vbm8sIG1vbm9zcGFjZSIgZm9udC1zaXplPSIxMC4wMCI+RW1pc3Npb248L3RleHQ+Cjwv
Zz4KPCEtLSBtYWNoaW5lIC0tPgo8ZyBpZD0ibm9kZTQiIGNsYXNzPSJub2RlIj4KPHRpdGxlPm1h
Y2hpbmU8L3RpdGxlPgo8ZWxsaXBzZSBmaWxsPSJub25lIiBzdHJva2U9ImJsYWNrIiBjeD0iMTYx
Ljg4IiBjeT0iLTExMiIgcng9IjE0Ny41MSIgcnk9IjE4Ii8+Cjx0ZXh0IHRleHQtYW5jaG9yPSJt
aWRkbGUiIHg9IjE2MS44OCIgeT0iLTEwOS41IiBmb250LWZhbWlseT0iQ291cmllciBOZXcsIERl
amFWdSBTYW5zIE1vbm8sIG1vbm9zcGFjZSIgZm9udC1zaXplPSIxMC4wMCI+TWFjaGluZSBDb2Rl
IChCaW5hcnkgSW5zdHJ1Y3Rpb24gV29yZHMpPC90ZXh0Pgo8L2c+CjwhLS0gYXNzZW1ibHkmIzQ1
OyZndDttYWNoaW5lIC0tPgo8ZyBpZD0iZWRnZTMiIGNsYXNzPSJlZGdlIj4KPHRpdGxlPmFzc2Vt
Ymx5JiM0NTsmZ3Q7bWFjaGluZTwvdGl0bGU+CjxwYXRoIGZpbGw9Im5vbmUiIHN0cm9rZT0iYmxh
Y2siIGQ9Ik0xNjEuODgsLTE3Ni44MkMxNjEuODgsLTE2Ni4xOSAxNjEuODgsLTE1Mi4zMSAxNjEu
ODgsLTE0MC4yIi8+Cjxwb2x5Z29uIGZpbGw9ImJsYWNrIiBzdHJva2U9ImJsYWNrIiBwb2ludHM9
IjE2NS4zOCwtMTQwLjE1IDE2MS44OCwtMTMwLjE1IDE1OC4zOCwtMTQwLjE1IDE2NS4zOCwtMTQw
LjE1Ii8+Cjx0ZXh0IHRleHQtYW5jaG9yPSJtaWRkbGUiIHg9IjIzNC4zOCIgeT0iLTE1MSIgZm9u
dC1mYW1pbHk9IkNvdXJpZXIgTmV3LCBEZWphVnUgU2FucyBNb25vLCBtb25vc3BhY2UiIGZvbnQt
c2l6ZT0iMTAuMDAiPiAmIzE2MDsmIzE2MDtBc3NlbWJsaW5nIC8gRW5jb2Rpbmc8L3RleHQ+Cjwv
Zz4KPCEtLSBoYXJkd2FyZSAtLT4KPGcgaWQ9Im5vZGU1IiBjbGFzcz0ibm9kZSI+Cjx0aXRsZT5o
YXJkd2FyZTwvdGl0bGU+CjxlbGxpcHNlIGZpbGw9Im5vbmUiIHN0cm9rZT0iYmxhY2siIGN4PSIx
NjEuODgiIGN5PSItMTgiIHJ4PSIxMzcuMzIiIHJ5PSIxOCIvPgo8dGV4dCB0ZXh0LWFuY2hvcj0i
bWlkZGxlIiB4PSIxNjEuODgiIHk9Ii0xNS41IiBmb250LWZhbWlseT0iQ291cmllciBOZXcsIERl
amFWdSBTYW5zIE1vbm8sIG1vbm9zcGFjZSIgZm9udC1zaXplPSIxMC4wMCI+SGFyZHdhcmU6IEFM
VSwgUmVnaXN0ZXJzLCBCdXNlcywgUkFNPC90ZXh0Pgo8L2c+CjwhLS0gbWFjaGluZSYjNDU7Jmd0
O2hhcmR3YXJlIC0tPgo8ZyBpZD0iZWRnZTQiIGNsYXNzPSJlZGdlIj4KPHRpdGxlPm1hY2hpbmUm
IzQ1OyZndDtoYXJkd2FyZTwvdGl0bGU+CjxwYXRoIGZpbGw9Im5vbmUiIHN0cm9rZT0iYmxhY2si
IGQ9Ik0xNjEuODgsLTkzLjdDMTYxLjg4LC04MC40NiAxNjEuODgsLTYxLjk1IDE2MS44OCwtNDYu
NjYiLz4KPHBvbHlnb24gZmlsbD0iYmxhY2siIHN0cm9rZT0iYmxhY2siIHBvaW50cz0iMTY1LjM4
LC00Ni4yMyAxNjEuODgsLTM2LjIzIDE1OC4zOCwtNDYuMjMgMTY1LjM4LC00Ni4yMyIvPgo8dGV4
dCB0ZXh0LWFuY2hvcj0ibWlkZGxlIiB4PSIyMjUuMzgiIHk9Ii02OCIgZm9udC1mYW1pbHk9IkNv
dXJpZXIgTmV3LCBEZWphVnUgU2FucyBNb25vLCBtb25vc3BhY2UiIGZvbnQtc2l6ZT0iMTAuMDAi
PiAmIzE2MDsmIzE2MDtFeGVjdXRpb24gb248L3RleHQ+Cjx0ZXh0IHRleHQtYW5jaG9yPSJtaWRk
bGUiIHg9IjIyNS4zOCIgeT0iLTU3IiBmb250LWZhbWlseT0iQ291cmllciBOZXcsIERlamFWdSBT
YW5zIE1vbm8sIG1vbm9zcGFjZSIgZm9udC1zaXplPSIxMC4wMCI+ICYjMTYwOyYjMTYwOyYjMTYw
O01pY3JvYXJjaGl0ZWN0dXJlPC90ZXh0Pgo8L2c+CjwvZz4KPC9zdmc+Cg==
" alt="" />
<figcaption>Figure 9–1: Levels of Abstraction</figcaption>
</figure>

To master systems programming in C++, we must master the lowest two
tiers of this stack.

### 9.2 The Von Neumann Architecture

Modern general-purpose computing is grounded in the architectural
paradigm formalised by John von Neumann in 1945. Prior to this design,
early computational engines (such as the ENIAC) were programmed by
physically rearranging patch cables and switches. The machine was
hardwired to perform a single computational task at a time; changing the
program meant rewiring the hardware.

The breakthrough of the **Von Neumann Architecture** (see Figure 9–2),
and its contemporary counterpart the *Turing Machine*, is the
**Stored-Program Concept**: *instructions and data share the same
addressable physical memory*.

<figure id="fig:stored-program">
<img src="data:image/svg+xml;base64,PHN2ZyB2aWV3Qm94PSIwIDIwIDkwMCA0NDAiIHdpZHRoPSIxMDAlIiBoZWlnaHQ9IjQyMCIgeG1s
bnM9Imh0dHA6Ly93d3cudzMub3JnLzIwMDAvc3ZnIj4KICA8ZGVmcz4KICAgIDxzdHlsZT4KICAg
ICAgOnJvb3QgewogICAgICAgIC0tc3ZnLWJnOiB2YXIoLS1iZywgI2ZmZmZmZik7CiAgICAgICAg
LS1zdmctZmc6IHZhcigtLWZnLCAjMDAwMDAwKTsKICAgICAgICAtLXN2Zy10aW50LTEtZmFsbGJh
Y2s6ICNjY2NjY2M7IC8qIDIwJSBibGFjayArIDgwJSB3aGl0ZSAqLwogICAgICAgIC0tc3ZnLXRp
bnQtMi1mYWxsYmFjazogIzk5OTk5OTsgLyogNDAlIGJsYWNrICsgNjAlIHdoaXRlICovCiAgICAg
ICAgLS1zdmctdGludC0xOiBjb2xvci1taXgoaW4gc3JnYiwgdmFyKC0tc3ZnLWZnKSAyMCUsIHZh
cigtLXN2Zy1iZykgODAlKTsKICAgICAgICAtLXN2Zy10aW50LTI6IGNvbG9yLW1peChpbiBzcmdi
LCB2YXIoLS1zdmctZmcpIDQwJSwgdmFyKC0tc3ZnLWJnKSA2MCUpOwogICAgICB9CiAgICAgIHRl
eHQgICB7IGZvbnQtZmFtaWx5OiAiQXJpYWwgTmFycm93IiwgRnJlZVNhbnMsIHNhbnMtc2VyaWY7
IH0KICAgIDwvc3R5bGU+CiAgICA8ZmlsdGVyIGlkPSJzaGFkb3ciIHg9Ii01JSIgeT0iLTUlIiB3
aWR0aD0iMTEwJSIgaGVpZ2h0PSIxMTAlIj4KICAgICAgPGZlRHJvcFNoYWRvdyBkeD0iMiIgZHk9
IjIiIHN0ZERldmlhdGlvbj0iMyIgZmxvb2Qtb3BhY2l0eT0iMC4xNSIvPgogICAgPC9maWx0ZXI+
CiAgICA8bWFya2VyIGlkPSJhcnJvdyIgdmlld0JveD0iMCAwIDEwIDEwIiByZWZYPSI2IiByZWZZ
PSI1IiBtYXJrZXJXaWR0aD0iNiIgbWFya2VySGVpZ2h0PSI2IiBvcmllbnQ9ImF1dG8tc3RhcnQt
cmV2ZXJzZSI+CiAgICAgIDxwYXRoIGQ9Ik0gMCAxIEwgMTAgNSBMIDAgOSB6IiBmaWxsPSJjb250
ZXh0LXN0cm9rZSIgLz4KICAgIDwvbWFya2VyPgogICAgPG1hcmtlciBpZD0iYXJyb3ctYmkiIHZp
ZXdCb3g9IjAgMCAxMCAxMCIgcmVmWD0iNSIgcmVmWT0iNSIgbWFya2VyV2lkdGg9IjYiIG1hcmtl
ckhlaWdodD0iNiIgb3JpZW50PSJhdXRvIj4KICAgICAgPHBhdGggZD0iTSAwIDUgTCA0IDIgTCA0
IDggeiBNIDEwIDUgTCA2IDIgTCA2IDggeiIgZmlsbD0iY29udGV4dC1zdHJva2UiIC8+CiAgICA8
L21hcmtlcj4KICA8L2RlZnM+CiAgPCEtLSBDUFUgQm94IE91dGVyIC0tPgogIDxyZWN0IHg9IjQw
IiB5PSI0MCIgd2lkdGg9IjQ2MCIgaGVpZ2h0PSI0MDAiIHJ4PSIxMiIgZmlsbD0idmFyKC0tc3Zn
LXRpbnQtMiwgdmFyKC0tc3ZnLXRpbnQtMi1mYWxsYmFjaykpIiBzdHJva2U9InZhcigtLXN2Zy1m
ZykiIHN0cm9rZS13aWR0aD0iMiIgZmlsdGVyPSJ1cmwoI3NoYWRvdykiLz4KICA8dGV4dCB4PSIy
NjAiIHk9Ijc1IiBmb250LXNpemU9IjE4IiBmb250LXdlaWdodD0iYm9sZCIgZmlsbD0idmFyKC0t
c3ZnLWZnKSIgdGV4dC1hbmNob3I9Im1pZGRsZSI+Q2VudHJhbCBQcm9jZXNzaW5nIFVuaXQgKENQ
VSk8L3RleHQ+CiAgPCEtLSBDb250cm9sIFVuaXQgLS0+CiAgPHJlY3QgeD0iNzAiIHk9IjEwMCIg
d2lkdGg9IjE5MCIgaGVpZ2h0PSIxNTAiIHJ4PSI4IiBmaWxsPSJ2YXIoLS1zdmctdGludC0xLCB2
YXIoLS1zdmctdGludC0xLWZhbGxiYWNrKSkiIHN0cm9rZT0idmFyKC0tc3ZnLWZnKSIgc3Ryb2tl
LXdpZHRoPSIxLjUiLz4KICA8dGV4dCB4PSI5MCIgeT0iMTMwIiBmb250LXNpemU9IjE0IiBmb250
LXdlaWdodD0iYm9sZCIgZmlsbD0idmFyKC0tc3ZnLWZnKSI+Q29udHJvbCBVbml0IChDVSk8L3Rl
eHQ+CiAgPHJlY3QgeD0iODUiIHk9IjE0NSIgd2lkdGg9IjE2MCIgaGVpZ2h0PSIzNiIgcng9IjQi
IGZpbGw9InZhcigtLXN2Zy10aW50LTEsIHZhcigtLXN2Zy10aW50LTEtZmFsbGJhY2spKSIgc3Ry
b2tlPSJ2YXIoLS1zdmctZmcpIi8+CiAgPHRleHQgeD0iOTUiIHk9IjE2OCIgZm9udC1zaXplPSIx
NCIgZmlsbD0idmFyKC0tc3ZnLWZnKSI+UHJvZ3JhbSBDb3VudGVyIChQQyk8L3RleHQ+CiAgPHJl
Y3QgeD0iODUiIHk9IjE5NSIgd2lkdGg9IjE2MCIgaGVpZ2h0PSIzNiIgcng9IjQiIGZpbGw9InZh
cigtLXN2Zy10aW50LTEsIHZhcigtLXN2Zy10aW50LTEtZmFsbGJhY2spKSIgc3Ryb2tlPSJ2YXIo
LS1zdmctZmcpIi8+CiAgPHRleHQgeD0iOTUiIHk9IjIxOCIgZm9udC1zaXplPSIxNCIgZmlsbD0i
dmFyKC0tc3ZnLWZnKSI+SW5zdHJ1Y3Rpb24gUmVnIChJUik8L3RleHQ+CiAgPCEtLSBBcml0aG1l
dGljIExvZ2ljIFVuaXQgLS0+CiAgPHJlY3QgeD0iMjg1IiB5PSIxMDAiIHdpZHRoPSIxOTAiIGhl
aWdodD0iMTUwIiByeD0iOCIgZmlsbD0idmFyKC0tc3ZnLXRpbnQtMSwgdmFyKC0tc3ZnLXRpbnQt
MS1mYWxsYmFjaykpIiBzdHJva2U9InZhcigtLXN2Zy1mZykiIHN0cm9rZS13aWR0aD0iMS41Ii8+
CiAgPHRleHQgeD0iMzA1IiB5PSIxMzAiIGZvbnQtc2l6ZT0iMTQiIGZvbnQtd2VpZ2h0PSJib2xk
IiBmaWxsPSJ2YXIoLS1zdmctZmcpIj5Bcml0aG1ldGljIExvZ2ljIFVuaXQ8L3RleHQ+CiAgPHBv
bHlnb24gcG9pbnRzPSIzMTAsMTUwIDQ1MCwxNTAgNDMwLDIyNSAzOTAsMjI1IDM4MCwyMTAgMzcw
LDIyNSAzMzAsMjI1IiBmaWxsPSJ2YXIoLS1zdmctdGludC0xLCB2YXIoLS1zdmctdGludC0xLWZh
bGxiYWNrKSkiIHN0cm9rZT0idmFyKC0tc3ZnLWZnKSIgc3Ryb2tlLXdpZHRoPSIxLjUiLz4KICA8
dGV4dCB4PSIzNjUiIHk9IjE4NSIgZm9udC1zaXplPSIxNCIgZm9udC13ZWlnaHQ9ImJvbGQiIGZp
bGw9InZhcigtLXN2Zy1mZykiPkFMVTwvdGV4dD4KICA8IS0tIFJlZ2lzdGVycyBCb3ggLS0+CiAg
PHJlY3QgeD0iNzAiIHk9IjI3NSIgd2lkdGg9IjQwNSIgaGVpZ2h0PSIxNDUiIHJ4PSI4IiBmaWxs
PSJ2YXIoLS1zdmctdGludC0xLCB2YXIoLS1zdmctdGludC0xLWZhbGxiYWNrKSkiIHN0cm9rZT0i
dmFyKC0tc3ZnLWZnKSIgc3Ryb2tlLXdpZHRoPSIxLjUiLz4KICA8dGV4dCB4PSI5MCIgeT0iMzA1
IiBmb250LXNpemU9IjE0IiBmb250LXdlaWdodD0iYm9sZCIgZmlsbD0idmFyKC0tc3ZnLWZnKSI+
UmVnaXN0ZXIgRmlsZSAmYW1wOyBTdGF0dXM8L3RleHQ+CiAgPHJlY3QgeD0iODUiIHk9IjMyNSIg
d2lkdGg9IjE4MCIgaGVpZ2h0PSIzNSIgcng9IjQiIGZpbGw9InZhcigtLXN2Zy10aW50LTEsIHZh
cigtLXN2Zy10aW50LTEtZmFsbGJhY2spKSIgc3Ryb2tlPSJ2YXIoLS1zdmctZmcpIi8+CiAgPHRl
eHQgeD0iOTUiIHk9IjM0OCIgZm9udC1zaXplPSIxNCIgZmlsbD0idmFyKC0tc3ZnLWZnKSI+QWNj
dW11bGF0b3IgKEFDQyAvIFIwKTwvdGV4dD4KICA8cmVjdCB4PSIyODAiIHk9IjMyNSIgd2lkdGg9
IjE4MCIgaGVpZ2h0PSIzNSIgcng9IjQiIGZpbGw9InZhcigtLXN2Zy10aW50LTEsIHZhcigtLXN2
Zy10aW50LTEtZmFsbGJhY2spKSIgc3Ryb2tlPSJ2YXIoLS1zdmctZmcpIi8+CiAgPHRleHQgeD0i
MjkwIiB5PSIzNDgiIGZvbnQtc2l6ZT0iMTQiIGZpbGw9InZhcigtLXN2Zy1mZykiPkdlbmVyYWwg
UmVnaXN0ZXJzIChSbik8L3RleHQ+CiAgPHJlY3QgeD0iODUiIHk9IjM3MCIgd2lkdGg9IjE4MCIg
aGVpZ2h0PSIzNSIgcng9IjQiIGZpbGw9InZhcigtLXN2Zy10aW50LTEsIHZhcigtLXN2Zy10aW50
LTEtZmFsbGJhY2spKSIgc3Ryb2tlPSJ2YXIoLS1zdmctZmcpIi8+CiAgPHRleHQgeD0iOTUiIHk9
IjM5MyIgZm9udC1zaXplPSIxNCIgZmlsbD0idmFyKC0tc3ZnLWZnKSI+TWVtb3J5IEFkZHIgUmVn
IChNQVIpPC90ZXh0PgogIDxyZWN0IHg9IjI4MCIgeT0iMzcwIiB3aWR0aD0iMTgwIiBoZWlnaHQ9
IjM1IiByeD0iNCIgZmlsbD0idmFyKC0tc3ZnLXRpbnQtMSwgdmFyKC0tc3ZnLXRpbnQtMS1mYWxs
YmFjaykpIiBzdHJva2U9InZhcigtLXN2Zy1mZykiLz4KICA8dGV4dCB4PSIyOTAiIHk9IjM5MyIg
Zm9udC1zaXplPSIxNCIgZmlsbD0idmFyKC0tc3ZnLWZnKSI+TWVtb3J5IERhdGEgUmVnIChNRFIp
PC90ZXh0PgogIDwhLS0gUkFNIEJveCBPdXRlciAtLT4KICA8cmVjdCB4PSI2NTAiIHk9IjQwIiB3
aWR0aD0iMjEwIiBoZWlnaHQ9IjQwMCIgcng9IjEyIiBmaWxsPSJ2YXIoLS1zdmctdGludC0yLCB2
YXIoLS1zdmctdGludC0yLWZhbGxiYWNrKSkiIHN0cm9rZT0idmFyKC0tc3ZnLWZnKSIgc3Ryb2tl
LXdpZHRoPSIyIiBmaWx0ZXI9InVybCgjc2hhZG93KSIvPgogIDx0ZXh0IHg9Ijc1NSIgeT0iNzUi
IGZvbnQtc2l6ZT0iMTgiIGZvbnQtd2VpZ2h0PSJib2xkIiBmaWxsPSJ2YXIoLS1zdmctZmcpIiB0
ZXh0LWFuY2hvcj0ibWlkZGxlIj5QcmltYXJ5IE1lbW9yeSAoUkFNKTwvdGV4dD4KICA8IS0tIFJB
TSBNZW1vcnkgQ2VsbHMgLS0+CiAgPGcgdHJhbnNmb3JtPSJ0cmFuc2xhdGUoNjcwLCAxMTUpIj4K
ICAgIDxyZWN0IHg9IjAiIHk9IjAiIHdpZHRoPSIxNzAiIGhlaWdodD0iMzUiIGZpbGw9InZhcigt
LXN2Zy10aW50LTEsIHZhcigtLXN2Zy10aW50LTEtZmFsbGJhY2spKSIgc3Ryb2tlPSJ2YXIoLS1z
dmctZmcpIiBzdHJva2UtZGFzaGFycmF5PSIyLDIiLz4KICAgIDx0ZXh0IHg9IjEwIiB5PSIyMiIg
Zm9udC1zaXplPSIxNCIgZmlsbD0idmFyKC0tc3ZnLWZnKSI+MHgwMDAwOiBMT0FEIDB4MDAxMDwv
dGV4dD4KICAgIDxyZWN0IHg9IjAiIHk9IjM1IiB3aWR0aD0iMTcwIiBoZWlnaHQ9IjM1IiBmaWxs
PSJ2YXIoLS1zdmctdGludC0xLCB2YXIoLS1zdmctdGludC0xLWZhbGxiYWNrKSkiIHN0cm9rZT0i
dmFyKC0tc3ZnLWZnKSIgc3Ryb2tlLWRhc2hhcnJheT0iMiwyIi8+CiAgICA8dGV4dCB4PSIxMCIg
eT0iNTciIGZvbnQtc2l6ZT0iMTQiIGZpbGw9InZhcigtLXN2Zy1mZykiPjB4MDAwMTogQUREICAw
eDAwMTE8L3RleHQ+CiAgICA8cmVjdCB4PSIwIiB5PSI3MCIgd2lkdGg9IjE3MCIgaGVpZ2h0PSIz
NSIgZmlsbD0idmFyKC0tc3ZnLXRpbnQtMSwgdmFyKC0tc3ZnLXRpbnQtMS1mYWxsYmFjaykpIiBz
dHJva2U9InZhcigtLXN2Zy1mZykiIHN0cm9rZS1kYXNoYXJyYXk9IjIsMiIvPgogICAgPHRleHQg
eD0iMTAiIHk9IjkyIiBmb250LXNpemU9IjE0IiBmaWxsPSJ2YXIoLS1zdmctZmcpIj4weDAwMDI6
IFNUT1JFIDB4MDAxMjwvdGV4dD4KICAgIDxyZWN0IHg9IjAiIHk9IjEwNSIgd2lkdGg9IjE3MCIg
aGVpZ2h0PSIzNSIgZmlsbD0idmFyKC0tc3ZnLXRpbnQtMSwgdmFyKC0tc3ZnLXRpbnQtMS1mYWxs
YmFjaykpIiBzdHJva2U9InZhcigtLXN2Zy1mZykiIHN0cm9rZS1kYXNoYXJyYXk9IjIsMiIvPgog
ICAgPHRleHQgeD0iMTAiIHk9IjEyNyIgZm9udC1zaXplPSIxNCIgZmlsbD0idmFyKC0tc3ZnLWZn
KSI+MHgwMDAzOiBKVU1QICAweDAwMDA8L3RleHQ+CiAgICA8cmVjdCB4PSIwIiB5PSIxNDAiIHdp
ZHRoPSIxNzAiIGhlaWdodD0iMzUiIGZpbGw9InZhcigtLXN2Zy10aW50LTEsIHZhcigtLXN2Zy10
aW50LTEtZmFsbGJhY2spKSIgc3Ryb2tlPSJ2YXIoLS1zdmctZmcpIi8+CiAgICA8dGV4dCB4PSIx
NSIgeT0iMTYyIiBmb250LXNpemU9IjE0IiBmaWxsPSJ2YXIoLS1zdmctZmcpIj4uLi4gY29kZSAv
IGRhdGEgLi4uPC90ZXh0PgogICAgPHJlY3QgeD0iMCIgeT0iMTc1IiB3aWR0aD0iMTcwIiBoZWln
aHQ9IjM1IiBmaWxsPSJ2YXIoLS1zdmctdGludC0xLCB2YXIoLS1zdmctdGludC0xLWZhbGxiYWNr
KSkiIHN0cm9rZT0idmFyKC0tc3ZnLWZnKSIvPgogICAgPHRleHQgeD0iMTAiIHk9IjE5NyIgZm9u
dC1zaXplPSIxNCIgZmlsbD0idmFyKC0tc3ZnLWZnKSI+MHgwMDEwOiAweDAwMkEgKDQyKTwvdGV4
dD4KICAgIDxyZWN0IHg9IjAiIHk9IjIxMCIgd2lkdGg9IjE3MCIgaGVpZ2h0PSIzNSIgZmlsbD0i
dmFyKC0tc3ZnLXRpbnQtMSwgdmFyKC0tc3ZnLXRpbnQtMS1mYWxsYmFjaykpIiBzdHJva2U9InZh
cigtLXN2Zy1mZykiLz4KICAgIDx0ZXh0IHg9IjEwIiB5PSIyMzIiIGZvbnQtc2l6ZT0iMTQiIGZp
bGw9InZhcigtLXN2Zy1mZykiPjB4MDAxMTogMHgwMDA4ICg4KTwvdGV4dD4KICAgIDxyZWN0IHg9
IjAiIHk9IjI0NSIgd2lkdGg9IjE3MCIgaGVpZ2h0PSIzNSIgZmlsbD0idmFyKC0tc3ZnLXRpbnQt
MSwgdmFyKC0tc3ZnLXRpbnQtMS1mYWxsYmFjaykpIiBzdHJva2U9InZhcigtLXN2Zy1mZykiLz4K
ICAgIDx0ZXh0IHg9IjEwIiB5PSIyNjciIGZvbnQtc2l6ZT0iMTQiIGZpbGw9InZhcigtLXN2Zy1m
ZykiPjB4MDAxMjogMHgwMDAwICgwKTwvdGV4dD4KICA8L2c+CiAgPCEtLSBCdXNlcyBDb25uZWN0
aW5nIENQVSBhbmQgUkFNIC0tPgogIDwhLS0gQWRkcmVzcyBCdXMgLS0+CiAgPHBhdGggZD0iTSA1
MDAgMzc1IEwgNjQ1IDM3NSIgc3Ryb2tlPSJ2YXIoLS1zdmctZmcpIiBzdHJva2Utd2lkdGg9IjIi
IG1hcmtlci1lbmQ9InVybCgjYXJyb3cpIi8+CiAgPHRleHQgeD0iNTc1IiB5PSIzNDUiIGZvbnQt
c2l6ZT0iMTQiIGZvbnQtd2VpZ2h0PSJib2xkIiBmaWxsPSJ2YXIoLS1zdmctZmcpIiB0ZXh0LWFu
Y2hvcj0ibWlkZGxlIj5BZGRyZXNzIEJ1czwvdGV4dD4KICA8dGV4dCB4PSI1NzUiIHk9IjM2NSIg
Zm9udC1zaXplPSIxNCIgZm9udC13ZWlnaHQ9ImJvbGQiIGZpbGw9InZhcigtLXN2Zy1mZykiIHRl
eHQtYW5jaG9yPSJtaWRkbGUiPihVbmlkaXJlY3Rpb25hbCk8L3RleHQ+CiAgPCEtLSBEYXRhIEJ1
cyAtLT4KICA8cGF0aCBkPSJNIDUwNSA0MDAgTCA2NDUgNDAwIiBzdHJva2U9InZhcigtLXN2Zy1m
ZykiIHN0cm9rZS13aWR0aD0iMiIgbWFya2VyLXN0YXJ0PSJ1cmwoI2Fycm93KSIgbWFya2VyLWVu
ZD0idXJsKCNhcnJvdykiLz4KICA8dGV4dCB4PSI1NzUiIHk9IjQyMCIgZm9udC1zaXplPSIxNCIg
Zm9udC13ZWlnaHQ9ImJvbGQiIGZpbGw9InZhcigtLXN2Zy1mZykiIHRleHQtYW5jaG9yPSJtaWRk
bGUiPkRhdGEgQnVzPC90ZXh0PgogIDx0ZXh0IHg9IjU3NSIgeT0iNDQwIiBmb250LXNpemU9IjE0
IiBmb250LXdlaWdodD0iYm9sZCIgZmlsbD0idmFyKC0tc3ZnLWZnKSIgdGV4dC1hbmNob3I9Im1p
ZGRsZSI+KEJpZGlyZWN0aW9uYWwpPC90ZXh0PgogIDwhLS0gQ29udHJvbCBCdXMgLS0+CiAgPHBh
dGggZD0iTSA1MDAgMTc1IEwgNjQ1IDE3NSIgc3Ryb2tlPSJ2YXIoLS1zdmctZmcpIiBzdHJva2Ut
d2lkdGg9IjIiIHN0cm9rZS1kYXNoYXJyYXk9IjQsNCIgbWFya2VyLWVuZD0idXJsKCNhcnJvdyki
Lz4KICA8dGV4dCB4PSI1NzUiIHk9IjE0NSIgZm9udC1zaXplPSIxNCIgZm9udC13ZWlnaHQ9ImJv
bGQiIGZpbGw9InZhcigtLXN2Zy1mZykiIHRleHQtYW5jaG9yPSJtaWRkbGUiPkNvbnRyb2wgQnVz
PC90ZXh0PgogIDx0ZXh0IHg9IjU3NSIgeT0iMTY1IiBmb250LXNpemU9IjE0IiBmb250LXdlaWdo
dD0iYm9sZCIgZmlsbD0idmFyKC0tc3ZnLWZnKSIgdGV4dC1hbmNob3I9Im1pZGRsZSI+KFJlYWQv
V3JpdGUpPC90ZXh0Pgo8L3N2Zz4=
" alt="Figure 9–2: Outline of the Von Neumann Architecture" />
<figcaption>Figure 9–2: Outline of the Von Neumann
Architecture</figcaption>
</figure>

#### 9.2.1 The System Subcomponents

A computer system consists of three foundational subsystems linked by
physical communication channels called **buses**:

1.  **The Central Processing Unit (CPU):** The brain of the
    architecture. It retrieves instructions, interprets their bit
    patterns, coordinates data movement, and performs computations.
2.  **Primary Storage (RAM):** A linearly addressed sequence of
    fixed-size storage cells. Crucially, RAM holds both the binary
    encoded instructions that constitute the program and the raw binary
    data operands manipulated by those instructions.
3.  **The System Buses:**
    - **Address Bus:** A unidirectional bus carrying the physical memory
      address that the CPU wishes to read from or write to. The bit
      width of this bus determines the maximum addressable memory
      capacity of the computer ($2^W$ locations for a $W$-bit address
      bus).
    - **Data Bus:** A bidirectional bus transferring raw bit words
      between the CPU registers and memory cells.
    - **Control Bus:** A collection of single-bit signal lines
      indicating operational states: Memory Read ($\text{MEMR}$), Memory
      Write ($\text{MEMW}$), Clock signals, and Hardware Interrupt
      requests.

#### 9.2.2 The Von Neumann Bottleneck

Because both instructions and data must traverse the same physical data
bus, the CPU cannot simultaneously read an instruction word and
read/write a data word in a single clock cycle. This architectural
throughput limit is known as the **Von Neumann Bottleneck**. Modern
architectures mitigate this through tiered on-chip SRAM caches (L1, L2,
L3) and separate L1 Instruction and Data caches (a hybrid model known as
the *Modified Harvard Architecture*).

### 9.3 Hardware Registers and CPU Microarchitecture

Registers are ultra-fast, single-word storage locations integrated
directly into the CPU datapath. Where access to off-chip DRAM can take
hundreds of CPU clock cycles, register access completes within a
fraction of a single clock cycle.

Registers fall into two distinct functional categories:

#### 9.3.1 Special-Purpose Architectural Registers

- **Program Counter (PC):** Holds the memory address of the *next*
  instruction word to be fetched. After every fetch operation, the PC
  automatically increments to point to the subsequent memory word,
  unless modified by a jump or branch instruction.
- **Instruction Register (IR):** Holds the binary bit pattern of the
  instruction currently being decoded and executed.
- **Memory Address Register (MAR):** Directly connected to the system
  Address Bus; latches the memory address for an imminent read or write
  transaction.
- **Memory Data Register (MDR) / Memory Buffer Register (MBR):**
  Connected directly to the system Data Bus. When writing to memory,
  data placed in the MDR is driven onto the bus; when reading from
  memory, data appearing on the bus is latched into the MDR.
- **Status / Flags Register (FLAGS):** A collection of individual 1-bit
  boolean flags updated automatically by the ALU following arithmetic
  and logical operations:
  - **Zero Flag ($Z$):** Set ($1$) if the result of the operation was
    exactly zero; cleared ($0$) otherwise.
  - **Sign / Negative Flag ($N$ or $S$):** Set if the high-order bit
    (MSB) of the result is $1$ (representing a negative value in two's
    complement).
  - **Carry Flag ($C$):** Set if an unsigned arithmetic operation
    produced an overflow out of the most significant bit.
  - **Overflow Flag ($V$ or $O$):** Set if a signed two's complement
    arithmetic operation yielded a result exceeding the register's
    representational boundaries.

#### 9.3.2 General-Purpose Registers and the Accumulator

Early microprocessors (such as the MOS 6502 or Intel 8080) and
educational models (such as the *Little Man Computer*) rely heavily on
an **Accumulator (ACC)** architecture. An accumulator is a default
register where the ALU implicitly deposits the result of every
calculation.

Modern architectures (x86-64, ARM, RISC-V) provide a bank of multiple
General-Purpose Registers (GPRs) (e.g., `RAX`, `RBX`, `RCX` on x86-64;
`X0` through `X30` on ARM64), enabling complex expressions to evaluate
entirely in on-chip silicon without spilling intermediate values back to
RAM.

### 9.4 The Fetch-Decode-Execute Cycle

The CPU operates in an endless, rhythmic state machine known as the
**Instruction Cycle** (or **Fetch-Decode-Execute Cycle**, see
Figure 9–3). Every single software instruction—from rendering a 3D
polygon to executing an HTTP request—is decomposed into these
fundamental atomic steps.

<figure id="fig:instruction-cycle">
<img src="data:image/svg+xml;base64,PHN2ZyB2aWV3Qm94PSIwIDIwIDkwMCAyMjAiIHdpZHRoPSIxMDAlIiBoZWlnaHQ9IjIyMCIgeG1s
bnM9Imh0dHA6Ly93d3cudzMub3JnLzIwMDAvc3ZnIj4KICA8ZGVmcz4KICAgIDxzdHlsZT4KICAg
ICAgOnJvb3QgewogICAgICAgIC0tc3ZnLWJnOiB2YXIoLS1iZywgI2ZmZmZmZik7CiAgICAgICAg
LS1zdmctZmc6IHZhcigtLWZnLCAjMDAwMDAwKTsKICAgICAgICAtLXN2Zy10aW50LTEtZmFsbGJh
Y2s6ICNjY2NjY2M7IC8qIDIwJSBibGFjayArIDgwJSB3aGl0ZSAqLwogICAgICAgIC0tc3ZnLXRp
bnQtMi1mYWxsYmFjazogIzk5OTk5OTsgLyogNDAlIGJsYWNrICsgNjAlIHdoaXRlICovCiAgICAg
ICAgLS1zdmctdGludC0xOiBjb2xvci1taXgoaW4gc3JnYiwgdmFyKC0tc3ZnLWZnKSAyMCUsIHZh
cigtLXN2Zy1iZykgODAlKTsKICAgICAgICAtLXN2Zy10aW50LTI6IGNvbG9yLW1peChpbiBzcmdi
LCB2YXIoLS1zdmctZmcpIDQwJSwgdmFyKC0tc3ZnLWJnKSA2MCUpOwogICAgICB9CiAgICAgIHRl
eHQgICB7IGZvbnQtZmFtaWx5OiAiQXJpYWwgTmFycm93IiwgRnJlZVNhbnMsIHNhbnMtc2VyaWY7
IH0KICAgIDwvc3R5bGU+CiAgICA8ZmlsdGVyIGlkPSJmZGUtc2hhZG93IiB4PSItNSUiIHk9Ii01
JSIgd2lkdGg9IjExMCUiIGhlaWdodD0iMTEwJSI+CiAgICAgIDxmZURyb3BTaGFkb3cgZHg9IjIi
IGR5PSIzIiBzdGREZXZpYXRpb249IjMiIGZsb29kLW9wYWNpdHk9IjAuMTIiLz4KICAgIDwvZmls
dGVyPgogICAgPG1hcmtlciBpZD0iZmRlLWFycm93IiB2aWV3Qm94PSIwIDAgMTAgMTAiIHJlZlg9
IjYiIHJlZlk9IjUiIG1hcmtlcldpZHRoPSI2IiBtYXJrZXJIZWlnaHQ9IjYiIG9yaWVudD0iYXV0
byI+CiAgICAgIDxwYXRoIGQ9Ik0gMCAxIEwgMTAgNSBMIDAgOSB6IiBmaWxsPSJjb250ZXh0LXN0
cm9rZSIvPgogICAgPC9tYXJrZXI+CiAgICA8bWFya2VyIGlkPSJmZGUtYXJyb3ctYmFjayIgdmll
d0JveD0iMCAwIDEwIDEwIiByZWZYPSI2IiByZWZZPSI1IiBtYXJrZXJXaWR0aD0iNiIgbWFya2Vy
SGVpZ2h0PSI2IiBvcmllbnQ9IjIxMCI+CiAgICAgIDxwYXRoIGQ9Ik0gMCAxIEwgMTAgNSBMIDAg
OSB6IiBmaWxsPSJjb250ZXh0LXN0cm9rZSIvPgogICAgPC9tYXJrZXI+CiAgPC9kZWZzPgogIDwh
LS0gU3RlcCAxOiBGZXRjaCAtLT4KICA8ZyBmaWx0ZXI9InVybCgjZmRlLXNoYWRvdykiPgogICAg
PHJlY3QgeD0iNDAiIHk9IjMwIiB3aWR0aD0iMjQwIiBoZWlnaHQ9IjE4MCIgcng9IjEwIiBmaWxs
PSJ2YXIoLS1zdmctdGludC0xLCB2YXIoLS1zdmctdGludC0xLWZhbGxiYWNrKSkiIHN0cm9rZT0i
dmFyKC0tc3ZnLWZnKSIgc3Ryb2tlLXdpZHRoPSIyIi8+CiAgPC9nPgogIDx0ZXh0IHg9IjE2MCIg
eT0iNTMiIGZvbnQtc2l6ZT0iMTQiIGZvbnQtd2VpZ2h0PSJib2xkIiBmaWxsPSJ2YXIoLS1zdmct
ZmcpIiB0ZXh0LWFuY2hvcj0ibWlkZGxlIj4xLiBGRVRDSDwvdGV4dD4KICA8dGV4dCB4PSI1NSIg
eT0iOTIiIGZvbnQtc2l6ZT0iMTQiIGZpbGw9InZhcigtLXN2Zy1mZykiPuKAoiBNQVIgJiM4NTky
OyBQQzwvdGV4dD4KICA8dGV4dCB4PSI1NSIgeT0iMTE3IiBmb250LXNpemU9IjE0IiBmaWxsPSJ2
YXIoLS1zdmctZmcpIj7igKIgTURSICYjODU5MjsgUkFNW01BUl08L3RleHQ+CiAgPHRleHQgeD0i
NTUiIHk9IjE0MiIgZm9udC1zaXplPSIxNCIgZmlsbD0idmFyKC0tc3ZnLWZnKSI+4oCiIElSICAm
Izg1OTI7IE1EUjwvdGV4dD4KICA8dGV4dCB4PSI1NSIgeT0iMTY3IiBmb250LXNpemU9IjE0IiBm
aWxsPSJ2YXIoLS1zdmctZmcpIj7igKIgUEMgICYjODU5MjsgUEMgKyAxPC90ZXh0PgogIDwhLS0g
QXJyb3cgMSB0byAyIC0tPgogIDxwYXRoIGQ9Ik0gMjg1IDEyMCBMIDMzMCAxMjAiIHN0cm9rZT0i
dmFyKC0tc3ZnLWZnKSIgc3Ryb2tlLXdpZHRoPSIzIiBtYXJrZXItZW5kPSJ1cmwoI2ZkZS1hcnJv
dykiLz4KICA8IS0tIFN0ZXAgMjogRGVjb2RlIC0tPgogIDxnIGZpbHRlcj0idXJsKCNmZGUtc2hh
ZG93KSI+CiAgICA8cmVjdCB4PSIzNDAiIHk9IjMwIiB3aWR0aD0iMjMwIiBoZWlnaHQ9IjE4MCIg
cng9IjEwIiBmaWxsPSJ2YXIoLS1zdmctdGludC0xLCB2YXIoLS1zdmctdGludC0xLWZhbGxiYWNr
KSkiIHN0cm9rZT0idmFyKC0tc3ZnLWZnKSIgc3Ryb2tlLXdpZHRoPSIyIi8+CiAgPC9nPgogIDx0
ZXh0IHg9IjQ1NSIgeT0iNTMiIGZvbnQtc2l6ZT0iMTQiIGZvbnQtd2VpZ2h0PSJib2xkIiBmaWxs
PSJ2YXIoLS1zdmctZmcpIiB0ZXh0LWFuY2hvcj0ibWlkZGxlIj4yLiBERUNPREU8L3RleHQ+CiAg
PHRleHQgeD0iMzU1IiB5PSI5MiIgZm9udC1zaXplPSIxNCIgZmlsbD0idmFyKC0tc3ZnLWZnKSI+
4oCiIENVIHJlYWRzIE9wY29kZSBpbiBJUjwvdGV4dD4KICA8dGV4dCB4PSIzNTUiIHk9IjExNyIg
Zm9udC1zaXplPSIxNCIgZmlsbD0idmFyKC0tc3ZnLWZnKSI+4oCiIFJlc29sdmVzIEFkZHJlc3Np
bmc8L3RleHQ+CiAgPHRleHQgeD0iMzU1IiB5PSIxNDIiIGZvbnQtc2l6ZT0iMTQiIGZpbGw9InZh
cigtLXN2Zy1mZykiPuKAoiBDb29yZGluYXRlcyBjb250cm9sPC90ZXh0PgogIDx0ZXh0IHg9IjM1
NSIgeT0iMTY3IiBmb250LXNpemU9IjE0IiBmaWxsPSJ2YXIoLS1zdmctZmcpIj4mI3hhMDsmI3hh
MDtsaW5lcyAmYW1wOyBBTFUgaW5wdXRzPC90ZXh0PgogIDwhLS0gQXJyb3cgMiB0byAzIC0tPgog
IDxwYXRoIGQ9Ik0gNTc1IDEyMCBMIDYyMCAxMjAiIHN0cm9rZT0idmFyKC0tc3ZnLWZnKSIgc3Ry
b2tlLXdpZHRoPSIzIiBtYXJrZXItZW5kPSJ1cmwoI2ZkZS1hcnJvdykiLz4KICA8IS0tIFN0ZXAg
MzogRXhlY3V0ZSAtLT4KICA8ZyBmaWx0ZXI9InVybCgjZmRlLXNoYWRvdykiPgogICAgPHJlY3Qg
eD0iNjMwIiB5PSIzMCIgd2lkdGg9IjIzMCIgaGVpZ2h0PSIxODAiIHJ4PSIxMCIgZmlsbD0idmFy
KC0tc3ZnLXRpbnQtMSwgdmFyKC0tc3ZnLXRpbnQtMS1mYWxsYmFjaykpIiBzdHJva2U9InZhcigt
LXN2Zy1mZykiIHN0cm9rZS13aWR0aD0iMiIvPgogIDwvZz4KICA8dGV4dCB4PSI3NDUiIHk9IjUz
IiBmb250LXNpemU9IjE0IiBmb250LXdlaWdodD0iYm9sZCIgZmlsbD0idmFyKC0tc3ZnLWZnKSIg
dGV4dC1hbmNob3I9Im1pZGRsZSI+My4gRVhFQ1VURTwvdGV4dD4KICA8dGV4dCB4PSI2NDUiIHk9
IjkyIiBmb250LXNpemU9IjE0IiBmaWxsPSJ2YXIoLS1zdmctZmcpIj7igKIgQUxVIHBlcmZvcm1z
IGNhbGM8L3RleHQ+CiAgPHRleHQgeD0iNjQ1IiB5PSIxMTciIGZvbnQtc2l6ZT0iMTQiIGZpbGw9
InZhcigtLXN2Zy1mZykiPuKAoiBGbGFncyB1cGRhdGVkIChaLE4sQyk8L3RleHQ+CiAgPHRleHQg
eD0iNjQ1IiB5PSIxNDIiIGZvbnQtc2l6ZT0iMTQiIGZpbGw9InZhcigtLXN2Zy1mZykiPuKAoiBS
ZWFkL1dyaXRlIFJBTSBpZiByZXEuPC90ZXh0PgogIDx0ZXh0IHg9IjY0NSIgeT0iMTY3IiBmb250
LXNpemU9IjE0IiBmaWxsPSJ2YXIoLS1zdmctZmcpIj7igKIgQnJhbmNoIG92ZXJyaWRlcyBQQzwv
dGV4dD4KICA8IS0tIEZlZWRiYWNrIExvb3AgUGF0aCAoRXhlY3V0ZSBiYWNrIHRvIEZldGNoKSAt
LT4KICA8cGF0aCBkPSJNIDc0NSAyMTAgQyA3NDUgMjM1LCAxNjAgMjM1LCAxNjAgMjE0IiBmaWxs
PSJub25lIiBzdHJva2U9InZhcigtLXN2Zy1mZykiIHN0cm9rZS13aWR0aD0iMiIgc3Ryb2tlLWRh
c2hhcnJheT0iNCw0IiBtYXJrZXItZW5kPSJ1cmwoI2ZkZS1hcnJvdy1iYWNrKSIvPgo8L3N2Zz4=
" alt="Figure 9–3: CPU Steps: Fetch, Decode, Execute" />
<figcaption>Figure 9–3: CPU Steps: Fetch, Decode, Execute</figcaption>
</figure>

Below is a detailed trace of the micro-operations occurring inside the
CPU at each phase:

#### Step 1: Fetch

1.  The memory address stored in the **Program Counter (PC)** is copied
    into the **Memory Address Register (MAR)**:
    $$\text{MAR} \leftarrow \text{PC}$$
2.  The Control Unit asserts the Memory Read line ($\text{MEMR}$). The
    RAM subsystem retrieves the bit word at $\text{MAR}$ and places it
    onto the Data Bus.
3.  The bit pattern on the Data Bus is latched into the **Memory Data
    Register (MDR)**: $$\text{MDR} \leftarrow \text{RAM}[\text{MAR}]$$
4.  The contents of the $\text{MDR}$ are copied into the **Instruction
    Register (IR)**: $$\text{IR} \leftarrow \text{MDR}$$
5.  The **Program Counter** is incremented by 1 (or the instruction
    length in bytes) so that it references the subsequent sequential
    instruction: $$\text{PC} \leftarrow \text{PC} + 1$$

#### Step 2: Decode

The Control Unit splits the instruction word in the $\text{IR}$ into its
constituent bitfields: \* **Operation Code (Opcode):** Identifies the
unique machine action to perform (e.g., `ADD`, `LOAD`, `JUMP`). \*
**Operand Field:** Contains immediate literal values, register
identifiers, or memory target addresses. \* Addressing modes are
resolved: Is the operand a direct memory address, an immediate numeric
constant, or an indirect pointer through a register?

#### Step 3: Execute

The hardware circuits enact the instruction:

- If the instruction is arithmetic (e.g., `ADD`), the Control Unit
  routes register values to the ALU inputs, signals the ALU to perform
  addition, latches the result into the Accumulator, and updates the
  `FLAGS` register.
- If the instruction is a memory store (e.g., `STA`), the target address
  is copied to $\text{MAR}$, the Accumulator is copied to $\text{MDR}$,
  and the Memory Write line ($\text{MEMW}$) is pulsed.
- If the instruction is a conditional branch (e.g., `JZ` - Jump if Zero)
  and the Zero Flag ($Z$) is set, the operand address is written
  directly into the **Program Counter**:
  $$\text{PC} \leftarrow \text{Operand Address}$$ This disrupts
  sequential linear execution, establishing loops and branching.

### 9.5 From High-Level Constructs to Assembly

Every high-level language structure compiles down to patterns of simple
arithmetic, register loads and stores, and conditional jumps. The next
part examines how the core primitives of structured programming map to
instruction sequences.

#### 9.5.1 Variables and Assignment

In high-level languages, such as C++, variable assignment is symbolic:

``` cpp
int x = 42;
int y = x + 10;
```

In an accumulator-based assembly language, this translates to distinct
memory offsets and register operations:

``` nasm
; Assume variable 'x' is mapped to memory address 0x0080
; Assume variable 'y' is mapped to memory address 0x0081

LDA_IMM 42       ; ACC <- 42 (Load immediate literal 42 into ACC)
STA     0x0080   ; RAM[0x0080] <- ACC (Store value of 'x')

LDA     0x0080   ; ACC <- RAM[0x0080] (Load value of 'x')
ADD_IMM 10       ; ACC <- ACC + 10
STA     0x0081   ; RAM[0x0081] <- ACC (Store into 'y')
```

#### 9.5.2 Conditional Branching (`if-else`)

High-level branching:

``` cpp
if (x == 0) {
    y = 1;
} else {
    y = 2;
}
```

Translates to a comparison, a flag evaluation, and conditional forward
jumps:

``` nasm
    LDA     0x0080      ; ACC <- x
    SUB_IMM 0           ; Compute ACC - 0 (sets Zero Flag if x == 0)
    JZ      THEN_BLOCK  ; If Zero Flag == 1, jump to THEN_BLOCK

ELSE_BLOCK:
    LDA_IMM 2           ; ACC <- 2
    STA     0x0081      ; y <- 2
    JMP     END_IF      ; Skip over the THEN_BLOCK

THEN_BLOCK:
    LDA_IMM 1           ; ACC <- 1
    STA     0x0081      ; y <- 1

END_IF:
    ; Execution continues...
```

#### 9.5.3 Conditional Iteration (`while` loop)

High-level loop:

``` cpp
int count = 5;
while (count > 0) {
    count = count - 1;
}
```

Translates to a loop header label, a termination check jumping past the
loop, and an unconditional jump returning to the header:

``` nasm
    LDA_IMM 5
    STA     0x0080      ; count = 5

LOOP_HEADER:
    LDA     0x0080      ; ACC <- count
    SUB_IMM 0           ; Check count against 0
    JZ      LOOP_EXIT   ; If count == 0, break loop
    JN      LOOP_EXIT   ; If count < 0, break loop

    ; Loop Body:
    LDA     0x0080
    SUB_IMM 1           ; count - 1
    STA     0x0080      ; count = count - 1

    JMP     LOOP_HEADER ; Unconditional backward jump

LOOP_EXIT:
    HLT                 ; Loop completed
```

### 9.6 Specification of the Virtual Architecture: The Mach-16

To ground these systems concepts without getting bogged down by the
immense complexity of modern x86-64 (which features thousands of pages
of legacy opcodes), we will formally design an 8-bit educational
architecture called the **Mach-16**.

We will specify its word size, register configuration, instruction
layout, and complete instruction set architecture (ISA). In the next
section, we will implement this entire architecture from scratch in C++.

<figure id="fig:instruction-word">
<img src="data:image/svg+xml;base64,PHN2ZyB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHdpZHRoPSI1MjIuNjAwMDQi
IGhlaWdodD0iOTcuNjAwMDA2IiBjbGFzcz0ic3ZnYm9iIiB2aWV3Qm94PSIwIC00IDUyMi42MDAw
NCA5Ny42MDAwMDYiPgogIDxzdHlsZT4uc3ZnYm9iIGxpbmUsIC5zdmdib2IgcGF0aCwgLnN2Z2Jv
YiBjaXJjbGUsIC5zdmdib2IgcmVjdCwgLnN2Z2JvYiBwb2x5Z29uIHsKICBzdHJva2U6IGJsYWNr
OwogIHN0cm9rZS13aWR0aDogMS41OwogIHN0cm9rZS1vcGFjaXR5OiAxOwogIGZpbGwtb3BhY2l0
eTogMTsKICBzdHJva2UtbGluZWNhcDogcm91bmQ7CiAgc3Ryb2tlLWxpbmVqb2luOiBtaXRlcjsK
fQoKLnN2Z2JvYiB0ZXh0IHsKICB3aGl0ZS1zcGFjZTogcHJlOwogIGZpbGw6IGJsYWNrOwogIGZv
bnQtZmFtaWx5OiBDb3VyaWVyIE5ldywgRGVqYVZ1IFNhbnMgTW9ubywgbW9ub3NwYWNlOwogIGZv
bnQtc2l6ZTogMTNweDsKfQoKLnN2Z2JvYiByZWN0LmJhY2tkcm9wIHsKICBzdHJva2U6IG5vbmU7
CiAgZmlsbDogdHJhbnNwYXJlbnQ7Cn0KCi5zdmdib2IgLmJyb2tlbiB7CiAgc3Ryb2tlLWRhc2hh
cnJheTogODsKfQoKLnN2Z2JvYiAuZmlsbGVkIHsKICBmaWxsOiBibGFjazsKfQoKLnN2Z2JvYiAu
YmdfZmlsbGVkIHsKICBmaWxsOiB0cmFuc3BhcmVudDsKICBzdHJva2Utd2lkdGg6IDE7Cn0KCi5z
dmdib2IgLm5vZmlsbCB7CiAgZmlsbDogdHJhbnNwYXJlbnQ7Cn0KCi5zdmdib2IgLmVuZF9tYXJr
ZWRfYXJyb3cgewogIG1hcmtlci1lbmQ6IHVybCgjYXJyb3cpOwp9Cgouc3ZnYm9iIC5zdGFydF9t
YXJrZWRfYXJyb3cgewogIG1hcmtlci1zdGFydDogdXJsKCNhcnJvdyk7Cn0KCi5zdmdib2IgLmVu
ZF9tYXJrZWRfZGlhbW9uZCB7CiAgbWFya2VyLWVuZDogdXJsKCNkaWFtb25kKTsKfQoKLnN2Z2Jv
YiAuc3RhcnRfbWFya2VkX2RpYW1vbmQgewogIG1hcmtlci1zdGFydDogdXJsKCNkaWFtb25kKTsK
fQoKLnN2Z2JvYiAuZW5kX21hcmtlZF9jaXJjbGUgewogIG1hcmtlci1lbmQ6IHVybCgjY2lyY2xl
KTsKfQoKLnN2Z2JvYiAuc3RhcnRfbWFya2VkX2NpcmNsZSB7CiAgbWFya2VyLXN0YXJ0OiB1cmwo
I2NpcmNsZSk7Cn0KCi5zdmdib2IgLmVuZF9tYXJrZWRfb3Blbl9jaXJjbGUgewogIG1hcmtlci1l
bmQ6IHVybCgjb3Blbl9jaXJjbGUpOwp9Cgouc3ZnYm9iIC5zdGFydF9tYXJrZWRfb3Blbl9jaXJj
bGUgewogIG1hcmtlci1zdGFydDogdXJsKCNvcGVuX2NpcmNsZSk7Cn0KCi5zdmdib2IgLmVuZF9t
YXJrZWRfYmlnX29wZW5fY2lyY2xlIHsKICBtYXJrZXItZW5kOiB1cmwoI2JpZ19vcGVuX2NpcmNs
ZSk7Cn0KCi5zdmdib2IgLnN0YXJ0X21hcmtlZF9iaWdfb3Blbl9jaXJjbGUgewogIG1hcmtlci1z
dGFydDogdXJsKCNiaWdfb3Blbl9jaXJjbGUpOwp9Cgo8L3N0eWxlPgogIDxkZWZzPgogICAgPG1h
cmtlciBpZD0iYXJyb3ciIHZpZXdCb3g9Ii0yIC0yIDggOCIgcmVmWD0iNCIgcmVmWT0iMiIgbWFy
a2VyV2lkdGg9IjciIG1hcmtlckhlaWdodD0iNyIgb3JpZW50PSJhdXRvLXN0YXJ0LXJldmVyc2Ui
PgogICAgICA8cG9seWdvbiBwb2ludHM9IjAsMCAwLDQgNCwyIDAsMCI+PC9wb2x5Z29uPgogICAg
PC9tYXJrZXI+CiAgICA8bWFya2VyIGlkPSJkaWFtb25kIiB2aWV3Qm94PSItMiAtMiA4IDgiIHJl
Zlg9IjQiIHJlZlk9IjIiIG1hcmtlcldpZHRoPSI3IiBtYXJrZXJIZWlnaHQ9IjciIG9yaWVudD0i
YXV0by1zdGFydC1yZXZlcnNlIj4KICAgICAgPHBvbHlnb24gcG9pbnRzPSIwLDIgMiwwIDQsMiAy
LDQgMCwyIj48L3BvbHlnb24+CiAgICA8L21hcmtlcj4KICAgIDxtYXJrZXIgaWQ9ImNpcmNsZSIg
dmlld0JveD0iMCAwIDggOCIgcmVmWD0iNCIgcmVmWT0iNCIgbWFya2VyV2lkdGg9IjciIG1hcmtl
ckhlaWdodD0iNyIgb3JpZW50PSJhdXRvLXN0YXJ0LXJldmVyc2UiPgogICAgICA8Y2lyY2xlIGN4
PSI0IiBjeT0iNCIgcj0iMiIgY2xhc3M9ImZpbGxlZCI+PC9jaXJjbGU+CiAgICA8L21hcmtlcj4K
ICAgIDxtYXJrZXIgaWQ9Im9wZW5fY2lyY2xlIiB2aWV3Qm94PSIwIDAgOCA4IiByZWZYPSI0IiBy
ZWZZPSI0IiBtYXJrZXJXaWR0aD0iNyIgbWFya2VySGVpZ2h0PSI3IiBvcmllbnQ9ImF1dG8tc3Rh
cnQtcmV2ZXJzZSI+CiAgICAgIDxjaXJjbGUgY3g9IjQiIGN5PSI0IiByPSIyIiBjbGFzcz0iYmdf
ZmlsbGVkIj48L2NpcmNsZT4KICAgIDwvbWFya2VyPgogICAgPG1hcmtlciBpZD0iYmlnX29wZW5f
Y2lyY2xlIiB2aWV3Qm94PSIwIDAgOCA4IiByZWZYPSI0IiByZWZZPSI0IiBtYXJrZXJXaWR0aD0i
NyIgbWFya2VySGVpZ2h0PSI3IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAgIDxj
aXJjbGUgY3g9IjQiIGN5PSI0IiByPSIzIiBjbGFzcz0iYmdfZmlsbGVkIj48L2NpcmNsZT4KICAg
IDwvbWFya2VyPgogIDwvZGVmcz4KICA8cmVjdCBjbGFzcz0iYmFja2Ryb3AiIHg9IjAiIHk9Ii00
IiB3aWR0aD0iNTIyLjYwMDA0IiBoZWlnaHQ9Ijk3LjYwMDAwNiI+PC9yZWN0PgogIDx0ZXh0IHht
bDpzcGFjZT0icHJlc2VydmUiIHg9IjU2LjU1MDAwMyIgeT0iMTEuNzAwMDAxIiA+MTU8L3RleHQ+
CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMTU3Ljk1IiB5PSIxMS43MDAwMDEiID4x
MjwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIxODEuMzUiIHk9IjExLjcw
MDAwMSIgPjExPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjIyMC4zNSIg
eT0iMTEuNzAwMDAxIiA+MTA8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0i
MjUxLjU1IiB5PSIxMS43MDAwMDEiID45PC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2Vy
dmUiIHg9IjQ5My4zNSIgeT0iMTEuNzAwMDAxIiA+MDwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9
InByZXNlcnZlIiB4PSI4Ny43NSIgeT0iNDIuOSIgPk9wY29kZTwvdGV4dD4KICA8dGV4dCB4bWw6
c3BhY2U9InByZXNlcnZlIiB4PSIxODkuMTUwMDEiIHk9IjQyLjkiID5Nb2RlPC90ZXh0PgogIDx0
ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjMxMy45NSIgeT0iNDIuOSIgPkFkZHJlc3MgLyBM
aXRlcmFsPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9Ijc5Ljk1MDAwNSIg
eT0iNTguNSIgPig0IGJpdHMpPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9
IjE4OS4xNTAwMSIgeT0iNTguNSIgPigyYik8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVz
ZXJ2ZSIgeD0iMzQ1LjE1IiB5PSI1OC41IiA+KDEwIGJpdHMpPC90ZXh0PgogIDxnPgogICAgPGxp
bmUgeDE9IjUwLjciIHkxPSIyMy40MDAwMDIiIHgyPSI1MTAuOTAwMDIiIHkyPSIyMy40MDAwMDIi
IGNsYXNzPSJzb2xpZCI+PC9saW5lPgogICAgPGxpbmUgeDE9IjUwLjciIHkxPSIyMy40MDAwMDIi
IHgyPSI1MC43IiB5Mj0iNzAuMjAwMDA1IiBjbGFzcz0ic29saWQiPjwvbGluZT4KICAgIDxsaW5l
IHgxPSIxNzUuNSIgeTE9IjIzLjQwMDAwMiIgeDI9IjE3NS41IiB5Mj0iNzAuMjAwMDA1IiBjbGFz
cz0ic29saWQiPjwvbGluZT4KICAgIDxsaW5lIHgxPSIyMzcuOTAwMDEiIHkxPSIyMy40MDAwMDIi
IHgyPSIyMzcuOTAwMDEiIHkyPSI3MC4yMDAwMDUiIGNsYXNzPSJzb2xpZCI+PC9saW5lPgogICAg
PGxpbmUgeDE9IjUxMC45MDAwMiIgeTE9IjIzLjQwMDAwMiIgeDI9IjUxMC45MDAwMiIgeTI9Ijcw
LjIwMDAwNSIgY2xhc3M9InNvbGlkIj48L2xpbmU+CiAgICA8bGluZSB4MT0iNTAuNyIgeTE9Ijcw
LjIwMDAwNSIgeDI9IjUxMC45MDAwMiIgeTI9IjcwLjIwMDAwNSIgY2xhc3M9InNvbGlkIj48L2xp
bmU+CiAgPC9nPgo8L3N2Zz4K
" alt="" />
<figcaption>Figure 9–4: 16-bit Instruction Word</figcaption>
</figure>

#### 9.6.1 Architectural Parameters

- **Memory Space:** 1024 addressable memory words ($0\text{x}0000$ to
  $0\text{x}03\text{FF}$).
- **Word Size:** 16-bit integer words (used for both instructions and
  data representations).
- **Endianness:** Native host endianness (simulated word-by-word).
- **Registers:**
  - `PC` (Program Counter): 10-bit unsigned register (points to
    addresses $0 \le \text{address} < 1024$).
  - `IR` (Instruction Register): 16-bit register holding the active
    instruction word.
  - `ACC` (Accumulator): 16-bit signed two's complement integer register
    (\$ -32,768 ,767 \$).
  - `FLAGS`:
    - `Z` (Zero Flag): Set when an arithmetic/logical result is `0`.
    - `N` (Negative Flag): Set when the sign bit (MSB) of `ACC` is `1`.
    - `C` (Carry Flag): Set when arithmetic overflow/carry occurs.
    - `H` (Halt Flag): Set when the machine executes the `HLT` opcode.

#### 9.6.2 Instruction Word Bitfield Encoding

Each instruction is encoded in a single 16-bit word, as shown in
Figure 9–4 and Table 9–1:

- **Bits \[15:12\] (4 bits):** Opcode (allowing up to 16 distinct
  operations).
- **Bits \[11:10\] (2 bits):** Addressing Mode:
  - `00` (Direct): The operand field is a direct memory address.
  - `01` (Immediate): The operand field is an unsigned 10-bit literal
    value.
  - `10` (Indirect): The operand field contains an address containing
    the target address.
- **Bits \[9:0\] (10 bits):** Operand value (addresses $0..1023$ or
  immediate values $0..1023$).

| Mnemonic | Opcode (Hex) | Opcode (Bin) | Description | Flags Affected |
|:--------------|:--------------|:--------------|:--------------|:--------------|
| `HLT` | `0x0` | `0000` | Halt machine execution. | `H = 1` |
| `NOP` | `0x1` | `0001` | No operation (idle 1 cycle). | None |
| `LDA` | `0x2` | `0010` | Load value into Accumulator: $\text{ACC} \leftarrow \text{Operand}$ | `Z, N` |
| `STA` | `0x3` | `0011` | Store Accumulator to memory: $\text{RAM}[\text{Address}] \leftarrow \text{ACC}$ | None |
| `ADD` | `0x4` | `0100` | Add to Accumulator: $\text{ACC} \leftarrow \text{ACC} + \text{Operand}$ | `Z, N, C` |
| `SUB` | `0x5` | `0101` | Subtract from Accumulator: $\text{ACC} \leftarrow \text{ACC} - \text{Operand}$ | `Z, N, C` |
| `MUL` | `0x6` | `0110` | Multiply: $\text{ACC} \leftarrow \text{ACC} \times \text{Operand}$ | `Z, N, C` |
| `DIV` | `0x7` | `0111` | Integer Division: $\text{ACC} \leftarrow \text{ACC} \div \text{Operand}$ | `Z, N` |
| `AND` | `0x8` | `1000` | Bitwise AND: $\text{ACC} \leftarrow \text{ACC} \land \text{Operand}$ | `Z, N` |
| `OR` | `0x9` | `1001` | Bitwise OR: $\text{ACC} \leftarrow \text{ACC} \lor \text{Operand}$ | `Z, N` |
| `XOR` | `0xA` | `1010` | Bitwise XOR: $\text{ACC} \leftarrow \text{ACC} \oplus \text{Operand}$ | `Z, N` |
| `NOT` | `0xB` | `1011` | Bitwise NOT: $\text{ACC} \leftarrow \sim\text{ACC}$ (Operand ignored) | `Z, N` |
| `JMP` | `0xC` | `1100` | Unconditional Jump: $\text{PC} \leftarrow \text{Address}$ | None |
| `JZ` | `0xD` | `1101` | Jump if Zero: If $Z == 1 \implies \text{PC} \leftarrow \text{Address}$ | None |
| `JNZ` | `0xE` | `1110` | Jump if Not Zero: If $Z == 0 \implies \text{PC} \leftarrow \text{Address}$ | None |
| `JN` | `0xF` | `1111` | Jump if Negative: If $N == 1 \implies \text{PC} \leftarrow \text{Address}$ | None |

Table 9–1: Instruction Set Architecture (ISA) {#tbl:instruction-set}

### 9.7 Modern C++ Systems Engineering Foundations

The Modern C++ features (C++20/C++23) that ensure clean, type-safe, and
zero-overhead systems programming and which are relevant to writing the
CPU emulator are explored next:

#### 9.7.1 Fixed-Width Integer Types and Type Safety

In low-level systems programming, standard types like `int` or `long`
are ambiguous because their bit width is platform-dependent. We
explicitly include `<cstdint>` and use exact-width types:

- `std::uint16_t` for 16-bit instruction words and memory cells.
- `std::int16_t` for signed accumulator arithmetic.
- `std::uint8_t` for byte manipulation and opcodes.

#### 9.7.2 Scoped Enumerations (`enum class`)

Rather than using raw C-style enums (which pollute the enclosing scope
and convert implicitly to integers), we use strongly typed `enum class`
definitions with explicit underlying types:

``` cpp
enum class Opcode : std::uint8_t {
    HLT = 0x0,
    NOP = 0x1,
    LDA = 0x2,
    STA = 0x3,
    ADD = 0x4,
    // ...
};
```

#### 9.7.3 `std::array`, `std::span`, and Boundary Guarantees

Rather than raw C-style arrays (`uint16_t mem[1024]`) or dynamically
allocated heap vectors (`std::vector<uint16_t>`), we use
`std::array<std::uint16_t, 1024>` for the machine's primary memory. This
provides value semantics, static size guarantees, cache locality, and
clean integration with standard algorithms and views (`std::span`).

### 9.8 Experimental Implementation: The Mach-16 Emulator in Modern C++

The listings below implement the complete emulator. The architecture is
divided cleanly into:

- Listing 9–1 contains instruction and types representation, being the
  definitions of opcodes, addressing modes, instruction words, and
  flags.
- Listing 9–3 builds parser that converts human-readable assembly source
  into binary machine words. Writing raw hexadecimal words by hand is
  error-prone, so a Two-Pass Assembler is implemented:
  - **Pass 1:** Scans the source text to identify labels (e.g., `LOOP:`,
    `DATA:`) and records their memory addresses in a **Symbol Table**.
  - **Pass 2:** Re-scans the text, translating mnemonics, resolving
    labels against the symbol table, and generating the final vector of
    binary words.
- Listing 9–2 contains the Virtual Machine: CPU state, bus structures,
  ALU logic, and the Fetch-Decode-Execute cycle loop. A complete Von
  Neumann state machine is simulated, complete with diagnostic telemetry
  that prints cycle-by-cycle register and bus interactions.
- Listing 9–4 assembles and runs a simple assembly-language script and
  outputs the result.

Listing 9–1: The Core Architecture Header (Mach-16.hpp):

In [ ]:
#ifndef MACH16_HPP
#define MACH16_HPP

#include <cstdint>
#include <array>
#include <string>
#include <vector>
#include <string_view>
#include <optional>
#include <iostream>
#include <iomanip>
#include <sstream>
#include <stdexcept>
#include <format>

namespace mach16 {

// The Mach-16 physical parameters
constexpr std::size_t MEMORY_SIZE = 1024; // 1024 16-bit words
constexpr std::uint16_t ADDRESS_MASK = 0x03FF; // 10-bit address space

enum class Opcode : std::uint8_t {
    HLT = 0x0,
    NOP = 0x1,
    LDA = 0x2,
    STA = 0x3,
    ADD = 0x4,
    SUB = 0x5,
    MUL = 0x6,
    DIV = 0x7,
    AND = 0x8,
    OR  = 0x9,
    XOR = 0xA,
    NOT = 0xB,
    JMP = 0xC,
    JZ  = 0xD,
    JNZ = 0xE,
    JN  = 0xF
};

enum class AddressingMode : std::uint8_t {
    Direct    = 0x0, // Address refers to memory location
    Immediate = 0x1, // Operand is a raw numeric literal
    Indirect  = 0x2  // Address refers to a pointer cell in memory
};

struct Flags {
    bool zero{false};
    bool negative{false};
    bool carry{false};
    bool halt{false};

    void reset() noexcept {
        zero = false;
        negative = false;
        carry = false;
        halt = false;
    }
};

// Decoded instruction representation
struct Instruction {
    Opcode opcode;
    AddressingMode mode;
    std::uint16_t operand; // 10 bits

    // Pack instruction into a raw 16-bit machine word
    [[nodiscard]] constexpr std::uint16_t encode() const noexcept {
        const auto op = static_cast<std::uint16_t>(opcode) & 0x0F;
        const auto md = static_cast<std::uint16_t>(mode) & 0x03;
        const auto opr = operand & ADDRESS_MASK;
        return static_cast<std::uint16_t>((op << 12) | (md << 10) | opr);
    }

    // Decode a raw 16-bit machine word into an Instruction
    [[nodiscard]] static constexpr Instruction decode(std::uint16_t word) noexcept {
        const auto op = static_cast<Opcode>((word >> 12) & 0x0F);
        const auto md = static_cast<AddressingMode>((word >> 10) & 0x03);
        const auto opr = static_cast<std::uint16_t>(word & ADDRESS_MASK);
        return Instruction{op, md, opr};
    }
};

// Convert opcode to human-readable string
[[nodiscard]] inline std::string_view opcode_to_string(Opcode op) noexcept {
    switch (op) {
        case Opcode::HLT: return "HLT";
        case Opcode::NOP: return "NOP";
        case Opcode::LDA: return "LDA";
        case Opcode::STA: return "STA";
        case Opcode::ADD: return "ADD";
        case Opcode::SUB: return "SUB";
        case Opcode::MUL: return "MUL";
        case Opcode::DIV: return "DIV";
        case Opcode::AND: return "AND";
        case Opcode::OR:  return "OR";
        case Opcode::XOR: return "XOR";
        case Opcode::NOT: return "NOT";
        case Opcode::JMP: return "JMP";
        case Opcode::JZ:  return "JZ";
        case Opcode::JNZ: return "JNZ";
        case Opcode::JN:  return "JN";
        default:          return "UNKNOWN";
    }
}

} // namespace mach16

#endif // MACH16_HPP

Listing 9–2: The Virtual Machine & CPU Emulator (Machine.hpp):

In [ ]:
#ifndef MACHINE_HPP
#define MACHINE_HPP

//#include "Mach-16.hpp"
#include <iostream>
#include <iomanip>

namespace mach16 {

class VirtualMachine {
public:
    VirtualMachine() noexcept {
        reset();
    }

    void reset() noexcept {
        m_memory.fill(0);
        m_pc = 0;
        m_ir = 0;
        m_acc = 0;
        m_mar = 0;
        m_mdr = 0;
        m_flags.reset();
        m_cycle_count = 0;
    }

    // Load a program into memory at a specified base offset
    void load_program(const std::vector<std::uint16_t>& program, std::uint16_t origin = 0) {
        if (origin + program.size() > MEMORY_SIZE) {
            throw std::out_of_range("Program binary exceeds physical memory bounds.");
        }
        for (std::size_t i = 0; i < program.size(); ++i) {
            m_memory[origin + i] = program[i];
        }
        m_pc = origin;
    }

    // Direct memory inspection/injection
    void write_memory(std::uint16_t address, std::uint16_t value) {
        m_memory[address & ADDRESS_MASK] = value;
    }

    [[nodiscard]] std::uint16_t read_memory(std::uint16_t address) const noexcept {
        return m_memory[address & ADDRESS_MASK];
    }

    // Execute a single Fetch-Decode-Execute cycle
    bool step(bool trace = false) {
        if (m_flags.halt) {
            return false;
        }

        // =========================================================
        // 1. FETCH STAGE
        // =========================================================
        m_mar = m_pc;
        m_mdr = m_memory[m_mar];
        m_ir = m_mdr;
        const std::uint16_t fetched_address = m_pc;
        m_pc = (m_pc + 1) & ADDRESS_MASK;

        // =========================================================
        // 2. DECODE STAGE
        // =========================================================
        const Instruction instr = Instruction::decode(m_ir);

        // Resolve operand according to Addressing Mode
        std::int16_t resolved_value = 0;
        std::uint16_t target_address = 0;

        if (instr.mode == AddressingMode::Immediate) {
            // Sign-extend 10-bit immediate if treating as signed integer
            // For simplicity, we zero-extend immediate literals 0..1023
            resolved_value = static_cast<std::int16_t>(instr.operand);
        } else if (instr.mode == AddressingMode::Direct) {
            target_address = instr.operand;
            m_mar = target_address;
            m_mdr = m_memory[m_mar];
            resolved_value = static_cast<std::int16_t>(m_mdr);
        } else if (instr.mode == AddressingMode::Indirect) {
            // Memory points to another address
            m_mar = instr.operand;
            m_mdr = m_memory[m_mar];
            target_address = m_mdr & ADDRESS_MASK;
            m_mar = target_address;
            m_mdr = m_memory[m_mar];
            resolved_value = static_cast<std::int16_t>(m_mdr);
        }

        if (trace) {
            log_trace(fetched_address, instr, resolved_value);
        }

        // =========================================================
        // 3. EXECUTE STAGE
        // =========================================================
        execute_instruction(instr, resolved_value, target_address);

        ++m_cycle_count;
        return !m_flags.halt;
    }

    // Run continuously until HLT or maximum cycle limit reached
    void run(std::size_t max_cycles = 1'000'000, bool trace = false) {
        while (!m_flags.halt && m_cycle_count < max_cycles) {
            step(trace);
        }
    }

    // Diagnostic Dump
    void dump_state() const {
        std::cout << "\n========== MACH-16 CPU STATE DUMP ==========\n";
        std::cout << std::format("Cycles Run : {}\n", m_cycle_count);
        std::cout << std::format("PC         : 0x{:04X} ({})\n", m_pc, m_pc);
        std::cout << std::format("IR         : 0x{:04X}\n", m_ir);
        std::cout << std::format("ACC (Hex)  : 0x{:04X}\n", static_cast<std::uint16_t>(m_acc));
        std::cout << std::format("ACC (Dec)  : {}\n", m_acc);
        std::cout << std::format("Flags      : [Z: {} | N: {} | C: {} | H: {}]\n",
                                 m_flags.zero ? 1 : 0,
                                 m_flags.negative ? 1 : 0,
                                 m_flags.carry ? 1 : 0,
                                 m_flags.halt ? 1 : 0);
        std::cout << "==========================================\n";
    }

    [[nodiscard]] std::int16_t get_acc() const noexcept { return m_acc; }
    [[nodiscard]] const Flags& get_flags() const noexcept { return m_flags; }
    [[nodiscard]] std::size_t get_cycle_count() const noexcept { return m_cycle_count; }

private:
    void update_arithmetic_flags(std::int32_t result) noexcept {
        m_acc = static_cast<std::int16_t>(result & 0xFFFF);
        m_flags.zero = (m_acc == 0);
        m_flags.negative = (m_acc < 0);
        m_flags.carry = (result > 32767 || result < -32768);
    }

    void update_logic_flags() noexcept {
        m_flags.zero = (m_acc == 0);
        m_flags.negative = (m_acc < 0);
        m_flags.carry = false; // Logic operations clear carry
    }

    void execute_instruction(const Instruction& instr, std::int16_t value, std::uint16_t target_addr) {
        switch (instr.opcode) {
            case Opcode::HLT:
                m_flags.halt = true;
                break;

            case Opcode::NOP:
                // No action performed
                break;

            case Opcode::LDA:
                m_acc = value;
                m_flags.zero = (m_acc == 0);
                m_flags.negative = (m_acc < 0);
                break;

            case Opcode::STA:
                m_mar = (instr.mode == AddressingMode::Indirect) ? target_addr : instr.operand;
                m_mdr = static_cast<std::uint16_t>(m_acc);
                m_memory[m_mar & ADDRESS_MASK] = m_mdr;
                break;

            case Opcode::ADD: {
                const std::int32_t res = static_cast<std::int32_t>(m_acc) + static_cast<std::int32_t>(value);
                update_arithmetic_flags(res);
                break;
            }

            case Opcode::SUB: {
                const std::int32_t res = static_cast<std::int32_t>(m_acc) - static_cast<std::int32_t>(value);
                update_arithmetic_flags(res);
                break;
            }

            case Opcode::MUL: {
                const std::int32_t res = static_cast<std::int32_t>(m_acc) * static_cast<std::int32_t>(value);
                update_arithmetic_flags(res);
                break;
            }

            case Opcode::DIV: {
                if (value == 0) {
                    throw std::runtime_error("Hardware Trap: Integer division by zero.");
                }
                const std::int32_t res = static_cast<std::int32_t>(m_acc) / static_cast<std::int32_t>(value);
                update_arithmetic_flags(res);
                break;
            }

            case Opcode::AND:
                m_acc = static_cast<std::int16_t>(m_acc & value);
                update_logic_flags();
                break;

            case Opcode::OR:
                m_acc = static_cast<std::int16_t>(m_acc | value);
                update_logic_flags();
                break;

            case Opcode::XOR:
                m_acc = static_cast<std::int16_t>(m_acc ^ value);
                update_logic_flags();
                break;

            case Opcode::NOT:
                m_acc = static_cast<std::int16_t>(~m_acc);
                update_logic_flags();
                break;

            case Opcode::JMP:
                m_pc = instr.operand & ADDRESS_MASK;
                break;

            case Opcode::JZ:
                if (m_flags.zero) {
                    m_pc = instr.operand & ADDRESS_MASK;
                }
                break;

            case Opcode::JNZ:
                if (!m_flags.zero) {
                    m_pc = instr.operand & ADDRESS_MASK;
                }
                break;

            case Opcode::JN:
                if (m_flags.negative) {
                    m_pc = instr.operand & ADDRESS_MASK;
                }
                break;
        }
    }

    void log_trace(std::uint16_t fetch_pc, const Instruction& instr, std::int16_t resolved_val) const {
        std::string mode_str;
        switch (instr.mode) {
            case AddressingMode::Direct:    mode_str = "[DIR]"; break;
            case AddressingMode::Immediate: mode_str = "[IMM]"; break;
            case AddressingMode::Indirect:  mode_str = "[IND]"; break;
        }

        std::cout << std::format("CYC:{:04d} | PC:0x{:03X} | RAW:0x{:04X} | {} {:<4} {:<4} (Val:{:>5}) | ACC:{:>6} | [Z:{} N:{} C:{}]\n",
            m_cycle_count,
            fetch_pc,
            m_ir,
            mode_str,
            opcode_to_string(instr.opcode),
            instr.operand,
            resolved_val,
            m_acc,
            m_flags.zero ? 1 : 0,
            m_flags.negative ? 1 : 0,
            m_flags.carry ? 1 : 0
        );
    }

    // Hardware State
    std::array<std::uint16_t, MEMORY_SIZE> m_memory{};
    std::uint16_t m_pc{0};
    std::uint16_t m_ir{0};
    std::int16_t  m_acc{0};
    std::uint16_t m_mar{0};
    std::uint16_t m_mdr{0};
    Flags         m_flags{};
    std::size_t   m_cycle_count{0};
};

} // namespace mach16

#endif // MACHINE_HPP

Listing 9–3: The Two-Pass Assembler (Assembler.hpp):

In [ ]:
#ifndef ASSEMBLER_HPP
#define ASSEMBLER_HPP

//#include "Mach-16.hpp"
#include <unordered_map>
#include <sstream>
#include <algorithm>

namespace mach16 {

class Assembler {
public:
    static std::vector<std::uint16_t> assemble(std::string_view source_code) {
        std::unordered_map<std::string, std::uint16_t> symbol_table;
        std::vector<std::string> lines = split_lines(source_code);

        // =========================================================
        // PASS 1: Symbol Identification
        // =========================================================
        std::uint16_t current_address = 0;
        for (const auto& raw_line : lines) {
            std::string line = strip_comments(raw_line);
            line = trim(line);
            if (line.empty()) continue;

            if (line.ends_with(':')) {
                // Label definition
                std::string label_name = line.substr(0, line.length() - 1);
                label_name = trim(label_name);
                if (symbol_table.contains(label_name)) {
                    throw std::runtime_error(std::format("Duplicate label: {}", label_name));
                }
                symbol_table[label_name] = current_address;
            } else {
                // Instruction or raw data directive
                ++current_address;
            }
        }

        // =========================================================
        // PASS 2: Code Generation
        // =========================================================
        std::vector<std::uint16_t> bytecode;
        current_address = 0;

        for (const auto& raw_line : lines) {
            std::string line = strip_comments(raw_line);
            line = trim(line);
            if (line.empty() || line.ends_with(':')) continue;

            std::stringstream ss(line);
            std::string mnemonic;
            ss >> mnemonic;
            std::transform(mnemonic.begin(), mnemonic.end(), mnemonic.begin(), ::toupper);

            // Special Pseudo-op: .WORD (stores a raw integer constant)
            if (mnemonic == ".WORD") {
                std::string literal_str;
                ss >> literal_str;
                std::uint16_t raw_val = parse_literal_or_symbol(literal_str, symbol_table);
                bytecode.push_back(raw_val);
                ++current_address;
                continue;
            }

            Opcode op = parse_opcode(mnemonic);
            AddressingMode mode = AddressingMode::Direct;
            std::uint16_t operand = 0;

            if (op == Opcode::HLT || op == Opcode::NOP || op == Opcode::NOT) {
                // No operands required
                bytecode.push_back(Instruction{op, AddressingMode::Direct, 0}.encode());
                ++current_address;
                continue;
            }

            std::string token;
            if (ss >> token) {
                if (token.starts_with('#')) {
                    mode = AddressingMode::Immediate;
                    operand = parse_literal_or_symbol(token.substr(1), symbol_table);
                } else if (token.starts_with('@')) {
                    mode = AddressingMode::Indirect;
                    operand = parse_literal_or_symbol(token.substr(1), symbol_table);
                } else {
                    mode = AddressingMode::Direct;
                    operand = parse_literal_or_symbol(token, symbol_table);
                }
            } else {
                throw std::runtime_error(std::format("Instruction {} requires an operand.", mnemonic));
            }

            bytecode.push_back(Instruction{op, mode, operand}.encode());
            ++current_address;
        }

        return bytecode;
    }

private:
    static std::vector<std::string> split_lines(std::string_view text) {
        std::vector<std::string> result;
        std::stringstream ss{std::string(text)};
        std::string line;
        while (std::getline(ss, line)) {
            result.push_back(line);
        }
        return result;
    }

    static std::string strip_comments(std::string_view line) {
        auto comment_pos = line.find(';');
        if (comment_pos != std::string_view::npos) {
            line = line.substr(0, comment_pos);
        }
        return std::string(line);
    }

    static std::string trim(std::string_view str) {
        const auto start = str.find_first_not_of(" \t\r\n");
        if (start == std::string_view::npos) return "";
        const auto end = str.find_last_not_of(" \t\r\n");
        return std::string(str.substr(start, end - start + 1));
    }

    static Opcode parse_opcode(std::string_view m) {
        if (m == "HLT") return Opcode::HLT;
        if (m == "NOP") return Opcode::NOP;
        if (m == "LDA") return Opcode::LDA;
        if (m == "STA") return Opcode::STA;
        if (m == "ADD") return Opcode::ADD;
        if (m == "SUB") return Opcode::SUB;
        if (m == "MUL") return Opcode::MUL;
        if (m == "DIV") return Opcode::DIV;
        if (m == "AND") return Opcode::AND;
        if (m == "OR")  return Opcode::OR;
        if (m == "XOR") return Opcode::XOR;
        if (m == "NOT") return Opcode::NOT;
        if (m == "JMP") return Opcode::JMP;
        if (m == "JZ")  return Opcode::JZ;
        if (m == "JNZ") return Opcode::JNZ;
        if (m == "JN")  return Opcode::JN;
        throw std::runtime_error(std::format("Unknown mnemonic: {}", m));
    }

    static std::uint16_t parse_literal_or_symbol(
        const std::string& token,
        const std::unordered_map<std::string, std::uint16_t>& symbols)
    {
        if (symbols.contains(token)) {
            return symbols.at(token);
        }
        try {
            if (token.starts_with("0x") || token.starts_with("0X")) {
                return static_cast<std::uint16_t>(std::stoul(token, nullptr, 16));
            }
            return static_cast<std::uint16_t>(std::stoul(token, nullptr, 10));
        } catch (...) {
            throw std::runtime_error(std::format("Failed to resolve symbol or literal: {}", token));
        }
    }
};

} // namespace mach16

#endif // ASSEMBLER_HPP

### 9.9 Comprehensive Trace Analysis: Computing Factorials

It is now possible to observe the virtual system in action. We will
assemble and execute a program that calculates the factorial of $5$
($5! = 120$).

#### 9.9.1 High-Level Logic and Assembly Source

The imperative logic:

``` cpp
int n = 5;
int result = 1;
while (n != 0) {
    result = result * n;
    n = n - 1;
}
```

Translates to the following Mach-16 Assembly program:

``` nasm
; ============================================
; Program: Factorial Calculation (5!)
; Expected Output: ACC = 120 (0x0078)
; ============================================

        LDA     VAL_N       ; ACC <- n (5)
        STA     COUNTER     ; COUNTER <- 5
        LDA_IMM #1          
        STA     RESULT      ; RESULT <- 1

LOOP:
        LDA     COUNTER     ; ACC <- COUNTER
        SUB     #1          ; ACC <- COUNTER - 1
        JZ      DONE        ; If COUNTER - 1 == 0, we are finished!
        
        LDA     RESULT      ; ACC <- RESULT
        MUL     COUNTER     ; ACC <- RESULT * COUNTER
        STA     RESULT      ; RESULT <- ACC

        LDA     COUNTER     ; ACC <- COUNTER
        SUB     #1          ; Decrement
        STA     COUNTER     ; Update COUNTER
        
        JMP     LOOP        ; Repeat

DONE:
        LDA     RESULT      ; Put final answer into ACC
        HLT                 ; Terminate machine

; Storage declarations
VAL_N:   .WORD 5
COUNTER: .WORD 0
RESULT:  .WORD 0
```

#### 9.9.2 Test Program and Build Script

Listing 9–4: Execution Driver Program (main.cpp):

In [ ]:
//#include "Mach-16.hpp"
//#include "Machine.hpp"
//#include "Assembler.hpp"
#include <iostream>

int main() {
    const std::string factorial_program = R"(
        LDA     VAL_N
        STA     COUNTER
        LDA     #1
        STA     RESULT

    LOOP:
        LDA     COUNTER
        SUB     #1
        JZ      DONE
        
        LDA     RESULT
        MUL     COUNTER
        STA     RESULT

        LDA     COUNTER
        SUB     #1
        STA     COUNTER
        
        JMP     LOOP

    DONE:
        LDA     RESULT
        HLT

    VAL_N:
        .WORD 5
    COUNTER:
        .WORD 0
    RESULT:
        .WORD 0
    )";

    try {
        std::cout << "--- Assembling Program ---\n";
        std::vector<std::uint16_t> binary = mach16::Assembler::assemble(factorial_program);
        std::cout << std::format("Assembled {} machine instructions/words.\n\n", binary.size());

        std::cout << "--- Executing on Mach-16 Virtual Machine (Cycle Trace) ---\n";
        mach16::VirtualMachine vm;
        vm.load_program(binary);
        
        // Run with live micro-architectural trace logging
        vm.run(1000, true);

        // Print final register and memory state
        vm.dump_state();

        std::cout << std::format("Verification: Factorial Result = {}\n", vm.get_acc());
        if (vm.get_acc() == 120) {
            std::cout << ">> SUCCESS: Result matches mathematical expectation (5! = 120).\n";
        } else {
            std::cerr << ">> ERROR: Unexpected computational result.\n";
        }

    } catch (const std::exception& e) {
        std::cerr << "Fatal Exception: " << e.what() << '\n';
        return 1;
    }

    return 0;
}

In [ ]:
main(); // Invocation of main program for Jupyter notebook running the xeus-cpp kernel

#### 9.9.3 Detailed Execution Trace Output

When compiled and run, the emulator yields the exact micro-architectural
telemetry showing each register transformation:

```
--- Assembling Program ---
Assembled 20 machine instructions/words.

--- Executing on Mach-16 Virtual Machine (Cycle Trace) ---
CYC:0000 | PC:0x000 | RAW:0x2011 | [DIR] LDA  17   (Val:    5) | ACC:     5 | [Z:0 N:0 C:0]
CYC:0001 | PC:0x001 | RAW:0x3012 | [DIR] STA  18   (Val:    0) | ACC:     5 | [Z:0 N:0 C:0]
CYC:0002 | PC:0x002 | RAW:0x2401 | [IMM] LDA  1    (Val:    1) | ACC:     1 | [Z:0 N:0 C:0]
CYC:0003 | PC:0x003 | RAW:0x3013 | [DIR] STA  19   (Val:    0) | ACC:     1 | [Z:0 N:0 C:0]
CYC:0004 | PC:0x004 | RAW:0x2012 | [DIR] LDA  18   (Val:    5) | ACC:     5 | [Z:0 N:0 C:0]
CYC:0005 | PC:0x005 | RAW:0x5401 | [IMM] SUB  1    (Val:    1) | ACC:     4 | [Z:0 N:0 C:0]
CYC:0006 | PC:0x006 | RAW:0xD00E | [DIR] JZ   14   (Val:    0) | ACC:     4 | [Z:0 N:0 C:0]
CYC:0007 | PC:0x007 | RAW:0x2013 | [DIR] LDA  19   (Val:    1) | ACC:     1 | [Z:0 N:0 C:0]
CYC:0008 | PC:0x008 | RAW:0x6012 | [DIR] MUL  18   (Val:    5) | ACC:     5 | [Z:0 N:0 C:0]
CYC:0009 | PC:0x009 | RAW:0x3013 | [DIR] STA  19   (Val:    1) | ACC:     5 | [Z:0 N:0 C:0]
CYC:0010 | PC:0x00A | RAW:0x2012 | [DIR] LDA  18   (Val:    5) | ACC:     5 | [Z:0 N:0 C:0]
CYC:0011 | PC:0x00B | RAW:0x5401 | [IMM] SUB  1    (Val:    1) | ACC:     4 | [Z:0 N:0 C:0]
CYC:0012 | PC:0x00C | RAW:0x3012 | [DIR] STA  18   (Val:    5) | ACC:     4 | [Z:0 N:0 C:0]
CYC:0013 | PC:0x00D | RAW:0xC004 | [DIR] JMP  4    (Val:    0) | ACC:     4 | [Z:0 N:0 C:0]
... [Cycles 14 through 45 iterate through COUNTER = 4, 3, 2, 1] ...
CYC:0046 | PC:0x005 | RAW:0x5401 | [IMM] SUB  1    (Val:    1) | ACC:     0 | [Z:1 N:0 C:0]
CYC:0047 | PC:0x006 | RAW:0xD00E | [DIR] JZ   14   (Val:    0) | ACC:     0 | [Z:1 N:0 C:0]
CYC:0048 | PC:0x00E | RAW:0x2013 | [DIR] LDA  19   (Val:  120) | ACC:   120 | [Z:0 N:0 C:0]
CYC:0049 | PC:0x00F | RAW:0x0000 | [DIR] HLT  0    (Val:    0) | ACC:   120 | [Z:0 N:0 C:0]

========== MACH-16 CPU STATE DUMP ==========
Cycles Run : 50
PC         : 0x0010 (16)
IR         : 0x0000
ACC (Hex)  : 0x0078
ACC (Dec)  : 120
Flags      : [Z: 0 | N: 0 | C: 0 | H: 1]
==========================================
Verification: Factorial Result = 120
>> SUCCESS: Result matches mathematical expectation (5! = 120).
```

Observe how at `CYC:0046`, `SUB #1` evaluates to `0`, setting the Zero
Flag (`Z:1`). On `CYC:0047`, `JZ 14` inspects the Zero Flag, observes it
is active, and overwrites the Program Counter (`PC`) with `14`
(`0x00E`), branching out of the loop and loading the final accumulated
result of `120`.

### 9.10 Summary

By stepping beneath the high-level compiler, we have replaced
abstractions with hardware mechanisms:

1.  **The Stored-Program Concept:** Instructions are simply binary
    numeric words resident in the same physical memory space as
    application data.
2.  **The CPU is a Hardware State Machine:** The CPU mechanically cycles
    through *Fetch* (transferring memory via the bus into the IR and
    incrementing PC), *Decode* (interpreting bit slices via the Control
    Unit), and *Execute* (routing signals through the ALU and asserting
    bus lines).
3.  **Structured Control Flow is Synthesised:** Loops, `if-else` blocks,
    and function returns are made possible by conditional jumps (`JZ`,
    `JN`) updating the Program Counter based on ALU arithmetic status
    flags (`Z`, `N`, `C`).
4.  **C++ Zero-Overhead Abstraction:** When we write Modern C++, the
    language abstractions (classes, strong types, templates, lambda
    expressions) are designed to collapse cleanly during compilation
    into this raw machine model without runtime performance penalties.

### 9.11 Programming Challenges

#### Exercise 9.1: Arithmetic and Logical Bitwise Shifting (`SHL`, `SHR`, `ASR`)

**Context and Theory:**

At the hardware level, bitwise shifts are among the fastest primitives
available in an Arithmetic Logic Unit (ALU). They serve two roles:
low-level bitfield unpacking and rapid mathematical multiplication or
division by powers of two ($x \times 2^k$ or $\lfloor x / 2^k \rfloor$).

A **Logical Shift Left (`SHL`)** shifts all bits towards the most
significant bit (MSB), shifting a zero into bit 0. The bit shifted out
of bit 15 is latched into the Carry Flag ($C$).

A **Logical Shift Right (`SHR`)** shifts all bits towards bit 0,
inserting a zero into bit 15 and placing the discarded bit 0 into the
Carry Flag.

An **Arithmetic Shift Right (`ASR`)**, by contrast, preserves the sign
bit (bit 15) to maintain two's complement signed integer invariants. If
bit 15 is `1` (negative), ones are shifted in from the left; if `0`,
zeroes are shifted in. Formally, for a 16-bit word
$W = [b_{15}, b_{14}, \dots, b_0]$:

$$\text{SHL}(W) = [b_{14}, b_{13}, \dots, b_0, 0], \quad C \leftarrow b_{15}$$
$$\text{SHR}(W) = [0, b_{15}, b_{14}, \dots, b_1], \quad C \leftarrow b_0$$
$$\text{ASR}(W) = [b_{15}, b_{15}, b_{14}, \dots, b_1], \quad C \leftarrow b_0$$

**Specification:**

Extend the Mach-16 instruction set to support `SHL`, `SHR`, and `ASR`.

1.  Use the operand field to indicate the shift magnitude (an immediate
    literal $k \in [1, 15]$).
2.  Update the Accumulator in place.
3.  Set the Carry Flag ($C$) to the value of the last bit shifted out.
4.  Update the Zero Flag ($Z$) and Negative Flag ($N$) based on the
    final Accumulator state.

``` cpp
// Skeleton: Add to execute_instruction in Machine.hpp
case Opcode::SHL: {
    const auto count = std::min<std::uint16_t>(instr.operand, 16);
    if (count > 0) {
        m_flags.carry = ((static_cast<std::uint16_t>(m_acc) >> (16 - count)) & 1) != 0;
        m_acc = static_cast<std::int16_t>(static_cast<std::uint16_t>(m_acc) << count);
        update_logic_flags();
    }
    break;
}
case Opcode::ASR: {
    const auto count = std::min<std::uint16_t>(instr.operand, 16);
    if (count > 0) {
        m_flags.carry = ((m_acc >> (count - 1)) & 1) != 0;
        m_acc = static_cast<std::int16_t>(m_acc >> count); // Implementation-defined sign extension
        update_logic_flags();
    }
    break;
}
```

**Verification Tasks:**

- **Task 1 (Unsigned Doubling and Carry):** Load `0x4001` into `ACC`.
  Execute `SHL #1`. Verify that `ACC == 0x8002`, $C = 0$, $N = 1$, and
  $Z = 0$. Execute `SHL #1` again. Verify that `ACC == 0x0004`, $C = 1$,
  $N = 0$, and $Z = 0$.
- **Task 2 (Signed Division by Four):** Load `-16` (`0xFFF0`) into
  `ACC`. Execute `ASR #2`. Verify that `ACC == -4` (`0xFFFC`), $C = 0$,
  $N = 1$, and $Z = 0$.
- **Task 3 (Logical Shift to Zero):** Load `0x0001` into `ACC`. Execute
  `SHR #1`. Verify that `ACC == 0x0000`, $C = 1$, $Z = 1$, and $N = 0$.

#### Exercise 9.2: Hardware Stack Architecture and Stack Pointer (`SP`)

**Context and Theory:**

General-purpose computers require a dynamic Last-In, First-Out (LIFO)
memory structure to handle expression evaluation, local variables, and
nested subroutine control flow. In hardware, this is governed by the
**Stack Pointer (`SP`)** register.

On most architectures (including x86 and ARM), the stack is
**descending**: it starts at the highest available RAM address and grows
downwards towards lower memory addresses.

- A `PUSH` operation decrements the stack pointer, then writes the data
  word to memory:
  $$\text{SP} \leftarrow \text{SP} - 1, \quad \text{RAM}[\text{SP}] \leftarrow \text{ACC}$$
- A `POP` operation reads the data word from memory, then increments the
  stack pointer:
  $$\text{ACC} \leftarrow \text{RAM}[\text{SP}], \quad \text{SP} \leftarrow \text{SP} + 1$$

Two catastrophic hardware conditions can arise:

1.  **Stack Overflow:** The stack grows downward so far that it collides
    with code or heap memory.
2.  **Stack Underflow:** A `POP` occurs when the stack is already empty
    ($\text{SP} > \text{STACK\_BASE}$).

**Specification:**

Add a 10-bit register `m_sp` to `VirtualMachine`, initialised to
`0x03FF` (top of memory). Define opcodes `PUSH` and `POP` (neither
requires an operand). Implement strict boundary verification: throw a
`std::runtime_error` on stack underflow
($\text{SP} > 0\text{x}03\text{FF}$) or if `SP` collides with the loaded
program space
($\text{SP} \le \text{Program Origin} + \text{Program Size}$).

``` cpp
// Additions to VirtualMachine in Machine.hpp
constexpr std::uint16_t STACK_BASE = 0x03FF;
std::uint16_t m_sp{STACK_BASE};

void execute_stack_op(Opcode op) {
    if (op == Opcode::PUSH) {
        if (m_sp == 0 || m_sp <= m_program_limit) 
            throw std::runtime_error("Hardware Fault: Stack Overflow");
        --m_sp;
        m_memory[m_sp] = static_cast<std::uint16_t>(m_acc);
    } else if (op == Opcode::POP) {
        if (m_sp >= STACK_BASE) 
            throw std::runtime_error("Hardware Fault: Stack Underflow");
        m_acc = static_cast<std::int16_t>(m_memory[m_sp]);
        ++m_sp;
        m_flags.zero = (m_acc == 0);
        m_flags.negative = (m_acc < 0);
    }
}
```

**Verification Tasks:**

- **Task 1 (LIFO Ordering):** Execute the sequence: `LDA #42`, `PUSH`,
  `LDA #99`, `PUSH`, `LDA #0`, `POP`, `STA 0x0100`, `POP`, `STA 0x0101`.
  Verify that RAM `0x0100 == 99`, RAM `0x0101 == 42`, and
  `SP == 0x03FF`.
- **Task 2 (Stack Underflow Detection):** Initialise the VM and
  immediately execute a single `POP` instruction without prior pushes.
  Verify that the VM aborts with a `"Hardware Fault: Stack Underflow"`
  exception.
- **Task 3 (Stack Overflow Boundary):** Set the program limit to
  `0x03FE`. Execute two consecutive `PUSH` instructions. Verify that the
  second `PUSH` triggers `"Hardware Fault: Stack Overflow"`.

#### Exercise 9.3: Subroutine Call and Return Mechanics (`CALL`, `RET`)

**Context and Theory:**

Higher-level procedural decomposition relies on subroutines (functions).
When a subroutine is called, the CPU must record the execution
context—specifically, the return address—so that it can resume
sequential execution once the subroutine completes.

A **`CALL <target>`** instruction executes two atomic actions:

1.  It pushes the address of the *subsequent* instruction ($\text{PC}$)
    onto the hardware stack:
    $$\text{SP} \leftarrow \text{SP} - 1, \quad \text{RAM}[\text{SP}] \leftarrow \text{PC}$$
2.  It transfers control by loading the subroutine entry address into
    the Program Counter: $$\text{PC} \leftarrow \text{target}$$

A **`RET`** instruction performs the inverse:

- It pops the saved return address from the stack into the Program
  Counter:
  $$\text{PC} \leftarrow \text{RAM}[\text{SP}], \quad \text{SP} \leftarrow \text{SP} + 1$$

Because return addresses are stacked in LIFO order, subroutines can be
nested arbitrarily deep (subject to available stack memory) and can even
call themselves recursively (see Figure 9–5).

<figure id="fig:subroutine">
<img src="data:image/svg+xml;base64,PHN2ZyB2aWV3Qm94PSI1MCAyMCA3ODAgMTYwIiB3aWR0aD0iMTAwJSIgaGVpZ2h0PSIxNjAiIHht
bG5zPSJodHRwOi8vd3d3LnczLm9yZy8yMDAwL3N2ZyI+CiAgPGRlZnM+CiAgICA8c3R5bGU+CiAg
ICAgIDpyb290IHsKICAgICAgICAtLXN2Zy1iZzogdmFyKC0tYmcsICNmZmZmZmYpOwogICAgICAg
IC0tc3ZnLWZnOiB2YXIoLS1mZywgIzAwMDAwMCk7CiAgICAgICAgLS1zdmctdGludC0xLWZhbGxi
YWNrOiAjY2NjY2NjOyAvKiAyMCUgYmxhY2sgKyA4MCUgd2hpdGUgKi8KICAgICAgICAtLXN2Zy10
aW50LTItZmFsbGJhY2s6ICM5OTk5OTk7IC8qIDQwJSBibGFjayArIDYwJSB3aGl0ZSAqLwogICAg
ICAgIC0tc3ZnLXRpbnQtMTogY29sb3ItbWl4KGluIHNyZ2IsIHZhcigtLXN2Zy1mZykgMjAlLCB2
YXIoLS1zdmctYmcpIDgwJSk7CiAgICAgICAgLS1zdmctdGludC0yOiBjb2xvci1taXgoaW4gc3Jn
YiwgdmFyKC0tc3ZnLWZnKSA0MCUsIHZhcigtLXN2Zy1iZykgNjAlKTsKICAgICAgfQogICAgICB0
ZXh0ICAgeyBmb250LWZhbWlseTogIkFyaWFsIE5hcnJvdyIsIEZyZWVTYW5zLCBzYW5zLXNlcmlm
OyB9CiAgICA8L3N0eWxlPgogICAgPG1hcmtlciBpZD0iY2FsbC1hcnJvdyIgdmlld0JveD0iMCAw
IDEwIDEwIiByZWZYPSI2IiByZWZZPSI1IiBtYXJrZXJXaWR0aD0iNiIgbWFya2VySGVpZ2h0PSI2
IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAgIDxwYXRoIGQ9Ik0gMCAxIEwgMTAg
NSBMIDAgOSB6IiBmaWxsPSJjb250ZXh0LXN0cm9rZSIvPgogICAgPC9tYXJrZXI+CiAgPC9kZWZz
PgogIDxyZWN0IHg9IjUwIiB5PSIzMCIgd2lkdGg9IjIyMCIgaGVpZ2h0PSIxNDAiIHJ4PSI4IiBm
aWxsPSJ2YXIoLS1zdmctdGludC0xLCB2YXIoLS1zdmctdGludC0xLWZhbGxiYWNrKSkiIHN0cm9r
ZT0idmFyKC0tc3ZnLWZnKSIvPgogIDx0ZXh0IHg9IjY1IiB5PSI1NSIgZm9udC1zaXplPSIxNCIg
Zm9udC13ZWlnaHQ9ImJvbGQiIGZpbGw9InZhcigtLXN2Zy1mZykiPk1haW4gQ29kZTwvdGV4dD4K
ICA8dGV4dCB4PSI2NSIgeT0iODUiIGZvbnQtc2l6ZT0iMTQiIGZpbGw9InZhcigtLXN2Zy1mZyki
PjB4MDA0OiBDQUxMIDB4MDIwPC90ZXh0PgogIDx0ZXh0IHg9IjY1IiB5PSIxMTUiIGZvbnQtc2l6
ZT0iMTQiIGZpbGw9InZhcigtLXN2Zy1mZykiPjB4MDA1OiBBREQgIzEgKFJldCk8L3RleHQ+CiAg
PHBhdGggZD0iTSAyNzAgODUgQyAzNDAgODUsIDM0MCA0NSwgNDE1IDQ1IiBmaWxsPSJub25lIiBz
dHJva2U9InZhcigtLXN2Zy1mZykiIHN0cm9rZS13aWR0aD0iMiIgbWFya2VyLWVuZD0idXJsKCNj
YWxsLWFycm93KSIvPgogIDxyZWN0IHg9IjQyMCIgeT0iMzAiIHdpZHRoPSIyMjAiIGhlaWdodD0i
MTQwIiByeD0iOCIgZmlsbD0idmFyKC0tc3ZnLXRpbnQtMSwgdmFyKC0tc3ZnLXRpbnQtMS1mYWxs
YmFjaykpIiBzdHJva2U9InZhcigtLXN2Zy1mZykiLz4KICA8dGV4dCB4PSI0MzUiIHk9IjU1IiBm
b250LXNpemU9IjE0IiBmb250LXdlaWdodD0iYm9sZCIgZmlsbD0idmFyKC0tc3ZnLWZnKSI+U3Vi
cm91dGluZSAoMHgwMjApPC90ZXh0PgogIDx0ZXh0IHg9IjQzNSIgeT0iODUiIGZvbnQtc2l6ZT0i
MTQiIGZpbGw9InZhcigtLXN2Zy1mZykiPjB4MDIwOiBNVUwgIzI8L3RleHQ+CiAgPHRleHQgeD0i
NDM1IiB5PSIxMTUiIGZvbnQtc2l6ZT0iMTQiIGZpbGw9InZhcigtLXN2Zy1mZykiPjB4MDIxOiBS
RVQ8L3RleHQ+CiAgPHBhdGggZD0iTSA0MjAgMTE1IEMgMzUwIDExNSwgMzUwIDExNSwgMjc1IDEx
NSIgZmlsbD0ibm9uZSIgc3Ryb2tlPSJ2YXIoLS1zdmctZmcpIiBzdHJva2Utd2lkdGg9IjIiIHN0
cm9rZS1kYXNoYXJyYXk9IjQsNCIgbWFya2VyLWVuZD0idXJsKCNjYWxsLWFycm93KSIvPgogIDxy
ZWN0IHg9IjY4MCIgeT0iMzAiIHdpZHRoPSIxMzAiIGhlaWdodD0iMTQwIiByeD0iOCIgZmlsbD0i
dmFyKC0tc3ZnLXRpbnQtMSwgdmFyKC0tc3ZnLXRpbnQtMS1mYWxsYmFjaykpIiBzdHJva2U9InZh
cigtLXN2Zy1mZykiLz4KICA8dGV4dCB4PSI2OTUiIHk9IjU1IiBmb250LXNpemU9IjE0IiBmb250
LXdlaWdodD0iYm9sZCIgZmlsbD0idmFyKC0tc3ZnLWZnKSI+U3RhY2s8L3RleHQ+CiAgPHRleHQg
eD0iNjk1IiB5PSI4NSIgZm9udC1zaXplPSIxNCIgZmlsbD0idmFyKC0tc3ZnLWZnKSI+U1Ag4oaS
IFsgMHgwMDUgXTwvdGV4dD4KPC9zdmc+
" alt="Figure 9–5: Subroutine Calls and Nesting" />
<figcaption>Figure 9–5: Subroutine Calls and Nesting</figcaption>
</figure>

**Specification:** Implement `CALL` and `RET` using the hardware stack
developed in Exercise 9.2. Update the assembler to resolve symbol
references for `CALL`. Ensure that when `CALL` executes, the pushed
address is the address of the instruction immediately following the
`CALL`.

``` cpp
// Add to execute_instruction in Machine.hpp
case Opcode::CALL:
    if (m_sp == 0 || m_sp <= m_program_limit)
        throw std::runtime_error("Stack Overflow on CALL");
    --m_sp;
    m_memory[m_sp] = m_pc; // m_pc has already incremented to the return address during fetch
    m_pc = (instr.mode == AddressingMode::Direct) ? instr.operand : m_memory[instr.operand];
    break;

case Opcode::RET:
    if (m_sp >= STACK_BASE)
        throw std::runtime_error("Stack Underflow on RET");
    m_pc = m_memory[m_sp] & ADDRESS_MASK;
    ++m_sp;
    break;
```

**Verification Tasks:**

- **Task 1 (Call/Return Flow):** Write a program where main code at
  address `0x000` loads `#5` into `ACC`, calls subroutine `DOUBLE_IT` at
  `0x010`, and then halts. `DOUBLE_IT` performs `ADD ACC` and `RET`.
  Verify that the VM terminates with `ACC == 10` and `PC == 0x002`.
- **Task 2 (Nested Subroutines):** Call subroutine `A`, which increments
  `ACC` and calls subroutine `B`. Subroutine `B` doubles `ACC` and calls
  `RET`. Subroutine `A` calls `RET`. Verify that both stack frames are
  unwound correctly and `SP` returns to `0x03FF`.
- **Task 3 (Stack Leak Detection):** Call a subroutine that pushes a
  value onto the stack but fails to pop it before executing `RET`.
  Verify that the VM returns to an invalid memory location or raises a
  stack fault.

#### Exercise 9.4: Indexed Base-Offset Addressing Mode

**Context and Theory:**

Linear array iteration requires variable memory addresses. Without an
**Indexed Addressing Mode**, a program must resort to self-modifying
code—dynamically computing instructions in RAM and rewriting their
operand fields before execution. Self-modifying code is dangerous, hard
to debug, and prohibited on modern architectures with Harvard
instruction-cache architectures.

Indexed addressing calculates the **Effective Address ($\text{EA}$)** at
runtime by adding the contents of a dedicated Index Register (`IX`) or
the Accumulator (`ACC`) to a constant base address specified in the
instruction operand:

$$\text{EA} = (\text{Base Address} + \text{Index}) \pmod{2^W}$$

This allows an array to remain fixed in memory while a loop increments
the offset register to access consecutive elements.

**Specification:**

Add an explicit 10-bit Index Register (`m_ix`) and an addressing mode
flag:

- Add `AddressingMode::Indexed` (encoded as binary `11` in bits
  \[11:10\]).
- Add instructions `LDX` (Load Index Register from memory/immediate) and
  `STX` (Store Index Register).
- When `AddressingMode::Indexed` is decoded for data instructions
  (`LDA`, `STA`, `ADD`, `SUB`), calculate:
  $$\text{EA} = (\text{instr.operand} + \text{m\_ix}) \ \& \ \text{ADDRESS\_MASK}$$
- Fetch or write using $\text{EA}$ as the target address.

``` cpp
// Add to operand resolution in Machine.hpp
else if (instr.mode == AddressingMode::Indexed) {
    target_address = (instr.operand + m_ix) & ADDRESS_MASK;
    m_mar = target_address;
    m_mdr = m_memory[m_mar];
    resolved_value = static_cast<std::int16_t>(m_mdr);
}
```

**Verification Tasks:**

- **Task 1 (Array Traversal):** Place five numbers
  `[10, 20, 30, 40, 50]` at memory address `0x0200`. Use a loop that
  initialises `m_ix = 0`, loads `0x0200, IX` into `ACC`, adds it to an
  accumulator sum, and increments `m_ix` until all elements are summed.
  Verify that the final sum in memory is `150`.
- **Task 2 (Indexed Store):** Write a loop that initialises a 10-element
  array starting at `0x0100` with the square of each index
  ($A[i] = i^2$) using `STA 0x0100, IX`. Verify that memory locations
  `0x0100` through `0x0109` contain
  `[0, 1, 4, 9, 16, 25, 36, 49, 64, 81]`.
- **Task 3 (Address Boundary Wrap):** Set base address to `0x03FE` and
  `IX` to `5`. Ensure the Effective Address wraps around correctly using
  `ADDRESS_MASK` ($\text{EA} = 0\text{x}0003$) without throwing an
  out-of-range exception.

#### Exercise 9.5: Signed Two's Complement Overflow Detection

**Context and Theory:**

In 16-bit signed arithmetic, values range from $-32,768$ (`0x8000`) to
$+32,767$ (`0x7FFF`). A **Carry** occurs when an unsigned addition
exceeds $2^{16} - 1$. An **Overflow**, however, occurs when the result
of a signed operation cannot be represented in 16-bit two's complement
form (see Figure 9–6).

The **Overflow Flag ($V$)** is governed by the sign bits of the operands
and the result:

- Adding two positive numbers must yield a positive number. If
  positive + positive = negative, an overflow has occurred.
- Adding two negative numbers must yield a negative number. If
  negative + negative = positive, an overflow has occurred.
- Adding a positive and a negative number can *never* produce an
  overflow.

Formally, letting $A_{15}$, $B_{15}$, and $R_{15}$ represent the sign
bits (bit 15) of operand $A$, operand $B$, and the computed result $R$:

$$V_{\text{ADD}} = (A_{15} \land B_{15} \land \neg R_{15}) \lor (\neg A_{15} \land \neg B_{15} \land R_{15})$$
$$V_{\text{SUB}} = (A_{15} \land \neg B_{15} \land \neg R_{15}) \lor (\neg A_{15} \land B_{15} \land R_{15})$$

<figure id="fig:valid-overflow">
<img src="data:image/svg+xml;base64,PHN2ZyB2aWV3Qm94PSI0MCAyMCA4MjAgMTQwIiB3aWR0aD0iMTAwJSIgaGVpZ2h0PSIxNDAiIHht
bG5zPSJodHRwOi8vd3d3LnczLm9yZy8yMDAwL3N2ZyI+CiAgPGRlZnM+CiAgICA8c3R5bGU+CiAg
ICAgIDpyb290IHsKICAgICAgICAtLXN2Zy1iZzogdmFyKC0tYmcsICNmZmZmZmYpOwogICAgICAg
IC0tc3ZnLWZnOiB2YXIoLS1mZywgIzAwMDAwMCk7CiAgICAgICAgLS1zdmctdGludC0xLWZhbGxi
YWNrOiAjY2NjY2NjOyAvKiAyMCUgYmxhY2sgKyA4MCUgd2hpdGUgKi8KICAgICAgICAtLXN2Zy10
aW50LTItZmFsbGJhY2s6ICM5OTk5OTk7IC8qIDQwJSBibGFjayArIDYwJSB3aGl0ZSAqLwogICAg
ICAgIC0tc3ZnLXRpbnQtMTogY29sb3ItbWl4KGluIHNyZ2IsIHZhcigtLXN2Zy1mZykgMjAlLCB2
YXIoLS1zdmctYmcpIDgwJSk7CiAgICAgICAgLS1zdmctdGludC0yOiBjb2xvci1taXgoaW4gc3Jn
YiwgdmFyKC0tc3ZnLWZnKSA0MCUsIHZhcigtLXN2Zy1iZykgNjAlKTsKICAgICAgfQogICAgICB0
ZXh0ICAgeyBmb250LWZhbWlseTogIkFyaWFsIE5hcnJvdyIsIEZyZWVTYW5zLCBzYW5zLXNlcmlm
OyB9CiAgICA8L3N0eWxlPgogIDwvZGVmcz4KICA8cmVjdCB4PSI1MCIgeT0iMzAiIHdpZHRoPSIz
NDAiIGhlaWdodD0iMTAwIiByeD0iOCIgZmlsbD0idmFyKC0tc3ZnLXRpbnQtMSwgdmFyKC0tc3Zn
LXRpbnQtMS1mYWxsYmFjaykpIiBzdHJva2U9InZhcigtLXN2Zy1mZykiLz4KICA8dGV4dCB4PSI3
MCIgeT0iNTAiIGZvbnQtc2l6ZT0iMTYiIGZvbnQtd2VpZ2h0PSJib2xkIiBmaWxsPSJ2YXIoLS1z
dmctZmcpIj5TaWduZWQgT3ZlcmZsb3cgKFYgPSAxKTwvdGV4dD4KICA8dGV4dCB4PSI3MCIgeT0i
NzUiIGZvbnQtc2l6ZT0iMTQiIGZpbGw9InZhcigtLXN2Zy1mZykiPjB4N0ZGRiAoKzMyNzY3KSAg
W1NpZ246IDBdPC90ZXh0PgogIDx0ZXh0IHg9IjcwIiB5PSI5NSIgZm9udC1zaXplPSIxNCIgZmls
bD0idmFyKC0tc3ZnLWZnKSI+KyAweDAwMDEgKCsxKSAgICAgIFtTaWduOiAwXTwvdGV4dD4KICA8
dGV4dCB4PSI3MCIgeT0iMTE1IiBmb250LXNpemU9IjE0IiBmaWxsPSJ2YXIoLS1zdmctZmcpIj49
IDB4ODAwMCAoLTMyNzY4KSAgW1NpZ246IDEhIEVycm9yXTwvdGV4dD4KICA8cmVjdCB4PSI0NjAi
IHk9IjMwIiB3aWR0aD0iMzQwIiBoZWlnaHQ9IjEwMCIgcng9IjgiIGZpbGw9InZhcigtLXN2Zy10
aW50LTEsIHZhcigtLXN2Zy10aW50LTEtZmFsbGJhY2spKSIgc3Ryb2tlPSJ2YXIoLS1zdmctZmcp
Ii8+CiAgPHRleHQgeD0iNDgwIiB5PSI1MCIgZm9udC1zaXplPSIxNiIgZm9udC13ZWlnaHQ9ImJv
bGQiIGZpbGw9InZhcigtLXN2Zy1mZykiPlVuc2lnbmVkIENhcnJ5IChDID0gMSwgViA9IDApPC90
ZXh0PgogIDx0ZXh0IHg9IjQ4MCIgeT0iNzQiIGZvbnQtc2l6ZT0iMTQiIGZpbGw9InZhcigtLXN2
Zy1mZykiPjB4RkZGRiAoNjU1MzUgLyAtMSkgW1NpZ246IDFdPC90ZXh0PgogIDx0ZXh0IHg9IjQ4
MCIgeT0iOTUiIGZvbnQtc2l6ZT0iMTQiIGZpbGw9InZhcigtLXN2Zy1mZykiPisgMHgwMDAxICgx
KSAgICAgICAgICBbU2lnbjogMF08L3RleHQ+CiAgPHRleHQgeD0iNDgwIiB5PSIxMTUiIGZvbnQt
c2l6ZT0iMTQiIGZpbGw9InZhcigtLXN2Zy1mZykiPj0gMHgwMDAwIChDYXJyeSBPdXQsIFZhbGlk
IFNpZ25lZCAwKTwvdGV4dD4KPC9zdmc+
" alt="Figure 9–6: Valid and Invalid Results" />
<figcaption>Figure 9–6: Valid and Invalid Results</figcaption>
</figure>

**Specification:**

1.  Add a boolean flag `overflow{false}` to the `Flags` struct.
2.  Update the `ADD` and `SUB` execution logic in `VirtualMachine` to
    compute both the unsigned Carry Flag ($C$) and the signed Overflow
    Flag ($V$) using bitwise equations.
3.  Introduce two new branch instructions: `JO` (Jump if Overflow,
    $V = 1$) and `JNO` (Jump if No Overflow, $V = 0$).

``` cpp
// Add to update_arithmetic_flags in Machine.hpp
void update_add_flags(std::int16_t a, std::int16_t b, std::int32_t raw_result) noexcept {
    m_acc = static_cast<std::int16_t>(raw_result & 0xFFFF);
    const bool a_sign = (a < 0);
    const bool b_sign = (b < 0);
    const bool r_sign = (m_acc < 0);

    m_flags.zero = (m_acc == 0);
    m_flags.negative = r_sign;
    m_flags.carry = (raw_result > 0xFFFF || raw_result < 0);
    m_flags.overflow = (!a_sign && !b_sign && r_sign) || (a_sign && b_sign && !r_sign);
}
```

**Verification Tasks:** \* **Task 1 (Positive Overflow):** Load `32767`
into `ACC`. Execute `ADD #1`. Verify that `ACC == -32768`, $V = 1$,
$C = 0$, and $N = 1$. Follow with `JO TARGET` and verify that the branch
is taken. \* **Task 2 (Negative Underflow):** Load `-32768` into `ACC`.
Execute `SUB #1`. Verify that `ACC == 32767`, $V = 1$, $C = 0$, and
$N = 0$. \* **Task 3 (Unsigned Carry Without Overflow):** Load `-1`
(`0xFFFF`) into `ACC`. Execute `ADD #1`. Verify that `ACC == 0`,
$C = 1$, $V = 0$, $Z = 1$, and $N = 0$.

#### Exercise 9.6: Memory-Mapped Input/Output (MMIO) Subsystem

**Context and Theory:**

CPUs interact with external hardware (keyboards, displays, disk
controllers, network cards) using one of two primary architectural
paradigms:

1.  **Isolated (Port-Mapped) I/O:** The CPU has dedicated instructions
    (`IN`, `OUT`) and a separate physical bus address space for
    peripheral devices.
2.  **Memory-Mapped I/O (MMIO):** Peripherals are wired directly to the
    system Address and Data buses. Specific regions of the standard
    memory address space are reserved as hardware registers.

Writing to a designated MMIO address writes directly to a device's
transmission buffer, and reading from an MMIO address inspects device
state. Because MMIO uses regular memory addresses, standard instructions
(`LDA`, `STA`) handle device communication without needing new opcodes.

**Specification:**

Designate the top two words of the Mach-16 address space as MMIO
registers:

- `0x03FE`: **KEYBOARD_IN** (Status & Data Register)
- `0x03FF`: **TERMINAL_OUT** (Text Display Register)

Modify `VirtualMachine::read_memory` and `VirtualMachine::write_memory`:

- When `write_memory` targets `0x03FF`, cast the low 8 bits of the value
  to `char` and stream it immediately to host `std::cout`.
- When `read_memory` targets `0x03FE`, fetch a single byte from a
  pre-buffered host input string (or return `0` if empty).

``` cpp
// Update memory interceptors in Machine.hpp
void write_memory(std::uint16_t address, std::uint16_t value) {
    const auto addr = address & ADDRESS_MASK;
    if (addr == 0x03FF) {
        std::cout << static_cast<char>(value & 0xFF) << std::flush;
        return;
    }
    m_memory[addr] = value;
}

std::uint16_t read_memory(std::uint16_t address) {
    const auto addr = address & ADDRESS_MASK;
    if (addr == 0x03FE) {
        return m_input_buffer.empty() ? 0 : pop_input_buffer();
    }
    return m_memory[addr];
}
```

**Verification Tasks:**

- **Task 1 (Character Emission):** Write a Mach-16 assembly routine that
  stores ASCII values for `'C'`, `'S'`, `'\n'` into `0x03FF`. Confirm
  that `"CS\n"` renders to the standard host terminal output.
- **Task 2 (Null-Terminated String Printer):** Write a subroutine
  `PRINT_STRING` that takes the starting address of a null-terminated
  string, loops through each character, writes it to `0x03FF`, and
  terminates upon reaching `0x0000`. Test with `"HELLO, MACH-16\n"`.
- **Task 3 (Echo Loop):** Feed the simulated keyboard buffer `"OK"`.
  Execute a polling loop that reads from `0x03FE` and writes each
  character back to `0x03FF` until a null character (`0`) is
  encountered. Verify that `"OK"` is mirrored to standard output.

#### Exercise 9.7: Interactive Step Debugger (GDB-Lite)

**Context and Theory:**

Software engineers rely on debuggers to inspect and control running
programs. A hardware debugger interacts with a CPU using three
fundamental primitives:

1.  **Breakpoints:** Setting an address intercept that halts the
    fetch-execute cycle right before the target instruction executes.
2.  **Single-Stepping:** Advancing execution by exactly one cycle,
    allowing developers to inspect register mutations.
3.  **State Inspection:** Dumping internal architectural registers
    (`PC`, `IR`, `ACC`, `FLAGS`, `SP`) and inspecting raw memory
    contents.

To implement software breakpoints without hardware traps, real-world
debuggers often temporarily replace the target instruction word in RAM
with a trap opcode (e.g., `INT 3` on x86, which is opcode `0xCC`),
restoring the original instruction once the breakpoint hits.

**Specification:**

Wrap the `VirtualMachine` in an interactive REPL class `Debugger`. The
debugger must intercept execution before the fetch stage of each cycle
and support the following commands:

- `break <hex_addr>`: Adds an address to the active breakpoint set.
- `step` (or `s`): Executes one machine cycle, prints the instruction
  executed, and displays the mutated registers.
- `continue` (or `c`): Resumes full-speed execution until a breakpoint
  is hit or the machine halts.
- `reg`: Dumps the current contents of `PC`, `ACC`, `IR`, `SP`, and
  flags.
- `examine <hex_addr> <count>` (or `x`): Dumps `<count>` memory words
  starting from `<hex_addr>`.

``` cpp
class Debugger {
public:
    explicit Debugger(VirtualMachine& vm) : m_vm(vm) {}
    void add_breakpoint(std::uint16_t addr) { m_breakpoints.insert(addr & ADDRESS_MASK); }
    void run_interactive();
private:
    VirtualMachine& m_vm;
    std::unordered_set<std::uint16_t> m_breakpoints;
};
```

**Verification Tasks:**

- **Task 1 (Breakpoint Interception):** Load the Factorial program from
  Chapter 9. Set a breakpoint at `LOOP` (`0x004`). Issue `continue`.
  Verify that the VM pauses at `PC == 0x004` and displays `ACC == 5`.
- **Task 2 (Single-Step Inspection):** Issue the `step` command three
  times sequentially from `0x004`. Verify that `PC` advances through
  `0x005`, `0x006`, and `0x007`, updating the Accumulator and flags at
  each step.
- **Task 3 (Memory Modification):** Pause at a breakpoint. Use a new
  command `set <addr> <val>` to alter the value of variable `COUNTER` in
  RAM from `5` to `2`. Continue execution and verify that the final
  result computed is $2! = 2$ instead of $5! = 120$.

#### Exercise 9.8: Two-Pass Macro Assembler

**Context and Theory:**

Assembly language provides a direct 1-to-1 translation between
human-readable mnemonics and binary machine instructions. However,
repeating boilerplate instruction sequences increases errors and bloats
code.

A **Macro** is an assembler directive that defines a reusable block of
assembly code. Unlike subroutines, which are called at runtime via
`CALL` and `RET`, macros are **expanded inline at assembly time** by
text substitution.

A macro preprocessor scans source text during a pre-assembly phase,
maintains a table of macro definitions, and substitutes macro
invocations with their underlying instruction stream, replacing formal
parameters with provided arguments.

``` nasm
; Definition:
%MACRO INCREMENT_MEM ADDR
    LDA ADDR
    ADD #1
    STA ADDR
%ENDMACRO

; Invocation:
INCREMENT_MEM 0x0080
```

**Specification:**

Extend the `Assembler` class from Chapter 9 to support `%MACRO` and
`%ENDMACRO` directives:

1.  Support parameterised arguments (e.g., `%MACRO ADD_TO DEST, VAL`).
2.  During Pass 1, strip macro definitions from the token stream and
    record them in an internal Macro Table.
3.  When a macro name is encountered in the instruction stream, expand
    it inline into its constituent lines, substituting arguments, before
    computing label addresses.

``` cpp
struct MacroDefinition {
    std::vector<std::string> parameters;
    std::vector<std::string> template_lines;
};
// Add to Assembler:
std::unordered_map<std::string, MacroDefinition> m_macro_table;
std::string expand_macros(std::string_view source);
```

**Verification Tasks:**

- **Task 1 (Parameterless Macro Expansion):** Define a macro
  `%MACRO CLEAR_ACC` that expands to `LDA #0`. Invoke it three times in
  an assembly program. Assemble the code and verify that the binary
  contains three identical `0x2400` words.

- **Task 2 (Multi-Parameter Macro):** Define `%MACRO TRANSFER SRC, DST`
  that expands to `LDA SRC` followed by `STA DST`. Assemble
  `TRANSFER 0x0050, 0x0060`. Verify that Pass 2 generates `LDA 0x0050`
  and `STA 0x0060` with correct direct addresses.

- **Task 3 (Label Resolution Across Macros):** Place a macro invocation
  directly before a label:

  ``` nasm
      CLEAR_ACC
  LOOP:
      JMP LOOP
  ```

  Verify that the label `LOOP` resolves to address `0x0001` (accounting
  for the macro expansion) rather than `0x0000`.

#### Exercise 9.9: Standalone Disassembler and Symbol Reconstructor

**Context and Theory:**

A **Disassembler** performs the inverse operation of an assembler: it
consumes raw binary machine words and reconstructs human-readable
assembly instructions.

This process faces a fundamental challenge: the **Von Neumann
equivalence of code and data**. Because both machine instructions and
numeric data constants are stored as identical 16-bit words in primary
memory, a disassembler cannot easily distinguish between a `0x4005` word
meant as `ADD 0x0005` and a `0x4005` word meant as the raw integer
`16389`.

Linear sweep disassemblers decode every memory cell sequentially. More
sophisticated recursive traversal disassemblers follow control flow
(`JMP`, `JZ`, `CALL`) starting from entry points, categorising reached
locations as instructions and unreached regions as data blocks
(`.WORD`).

**Specification:**

Write a standalone class `Disassembler` that accepts a
`std::span<const std::uint16_t>` binary image and generates a valid
assembly source string:

1.  Decode each 16-bit word into its 4-bit Opcode, 2-bit Addressing
    Mode, and 10-bit Operand.
2.  Format addressing modes correctly (`#` for Immediate, `@` for
    Indirect, bare address for Direct).
3.  Use control flow analysis to synthesise automatic label names (`L0`,
    `L1`) for all jump targets.

``` cpp
class Disassembler {
public:
    [[nodiscard]] static std::string disassemble(std::span<const std::uint16_t> binary) {
        std::stringstream ss;
        for (std::size_t i = 0; i < binary.size(); ++i) {
            const auto instr = Instruction::decode(binary[i]);
            ss << std::format("0x{:03X}:  {:<5} ", i, opcode_to_string(instr.opcode));
            // Format operands based on addressing mode...
        }
        return ss.str();
    }
};
```

**Verification Tasks:**

- **Task 1 (Instruction Reconstruction):** Pass the binary array
  `{0x2405, 0x4402, 0x3010, 0x0000}` to the disassembler. Verify that
  the output string matches:
  `plaintext   0x000: LDA #5   0x001: ADD #2   0x002: STA 16   0x003: HLT`
- **Task 2 (Round-Trip Fidelity):** Assemble a source program containing
  arithmetic, loops, and conditional jumps. Disassemble the resulting
  binary back to assembly text. Re-assemble that generated text. Verify
  that the second binary output is bit-for-bit identical to the first.
- **Task 3 (Indirect Mode Disassembly):** Pass the raw word `0x280A`
  (Opcode `0x2` \[LDA\], Mode `0x2` \[Indirect\], Operand `10`). Verify
  that the disassembler outputs `LDA @10`.

#### Exercise 9.10: Fibonacci Sequence Machine Code Generator

**Context and Theory:**

The Fibonacci sequence is defined by the recurrence relation:

$$F(0) = 0, \quad F(1) = 1, \quad F(n) = F(n-1) + F(n-2) \quad \text{for } n \ge 2$$

On an accumulator-based architecture like the Mach-16, computing this
sequence exposes the limitations of having only one general-purpose
working register. Because the ALU cannot hold both $F(n-1)$ and $F(n-2)$
in silicon simultaneously, the program must frequently spill
intermediate results to scratchpad memory locations in RAM using `STA`
and reload them using `LDA`.

**Specification:**

Write a complete, optimised Mach-16 assembly language program that
computes the first 10 Fibonacci numbers
($0, 1, 1, 2, 3, 5, 8, 13, 21, 34$) and stores them sequentially into a
dedicated RAM buffer starting at address `0x0100`.

Requirements:

1.  The program must be purely iterative (using a loop counter
    decremented to zero).
2.  Use indexed addressing (`STA 0x0100, IX`) or indirect pointer
    updates to store terms into the output array.
3.  The program must terminate cleanly with `HLT`.

``` nasm
; Program Structure Sketch
        LDA #0
        STA 0x0100       ; F(0) = 0
        LDA #1
        STA 0x0101       ; F(1) = 1
        LDX #2           ; Index = 2
FIB_LOOP:
        ; Compute F(n) = F(n-1) + F(n-2)
        ; Store to 0x0100, IX
        ; Check if IX == 10
        ; Loop until done
        HLT
```

**Verification Tasks:**

- **Task 1 (Array Verification):** Assemble and run the program on the
  `VirtualMachine`. Verify that memory locations `0x0100` through
  `0x0109` contain exactly `{0, 1, 1, 2, 3, 5, 8, 13, 21, 34}`.
- **Task 2 (Register State on Termination):** Verify that upon halting,
  `PC` points to the `HLT` instruction, the Zero Flag ($Z$) is set to
  `1` by the loop termination check, and no memory outside of
  `0x0100`–`0x0109` and designated scratchpad variables was modified.
- **Task 3 (Cycle Efficiency):** Profile the execution. Verify that the
  algorithm computes all 10 terms in under 200 total clock cycles.

#### Exercise 9.11: 3-Stage Pipeline Simulation & Branch Hazard Unit

**Context and Theory:**

The classic Von Neumann machine runs strictly sequentially: an
instruction completes its Fetch, Decode, and Execute stages before the
next instruction's Fetch begins. This results in poor hardware
utilisation, as the ALU sits idle during Fetch, and the memory bus sits
idle during Execute.

Modern high-performance microprocessors use **Instruction Pipelining**.
A 3-stage pipeline divides execution into three concurrent stages
running in lockstep, as in Figure 9–7:

<figure id="fig:instruction-pipelining">
<img src="data:image/svg+xml;base64,PHN2ZyB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHdpZHRoPSI0NjAuMiIgaGVp
Z2h0PSI4MiIgY2xhc3M9InN2Z2JvYiIgdmlld0JveD0iMCAtNCA0NjAuMiA4MiI+CiAgPHN0eWxl
Pi5zdmdib2IgbGluZSwgLnN2Z2JvYiBwYXRoLCAuc3ZnYm9iIGNpcmNsZSwgLnN2Z2JvYiByZWN0
LCAuc3ZnYm9iIHBvbHlnb24gewogIHN0cm9rZTogYmxhY2s7CiAgc3Ryb2tlLXdpZHRoOiAxLjU7
CiAgc3Ryb2tlLW9wYWNpdHk6IDE7CiAgZmlsbC1vcGFjaXR5OiAxOwogIHN0cm9rZS1saW5lY2Fw
OiByb3VuZDsKICBzdHJva2UtbGluZWpvaW46IG1pdGVyOwp9Cgouc3ZnYm9iIHRleHQgewogIHdo
aXRlLXNwYWNlOiBwcmU7CiAgZmlsbDogYmxhY2s7CiAgZm9udC1mYW1pbHk6IENvdXJpZXIgTmV3
LCBEZWphVnUgU2FucyBNb25vLCBtb25vc3BhY2U7CiAgZm9udC1zaXplOiAxM3B4Owp9Cgouc3Zn
Ym9iIHJlY3QuYmFja2Ryb3AgewogIHN0cm9rZTogbm9uZTsKICBmaWxsOiB0cmFuc3BhcmVudDsK
fQoKLnN2Z2JvYiAuYnJva2VuIHsKICBzdHJva2UtZGFzaGFycmF5OiA4Owp9Cgouc3ZnYm9iIC5m
aWxsZWQgewogIGZpbGw6IGJsYWNrOwp9Cgouc3ZnYm9iIC5iZ19maWxsZWQgewogIGZpbGw6IHRy
YW5zcGFyZW50OwogIHN0cm9rZS13aWR0aDogMTsKfQoKLnN2Z2JvYiAubm9maWxsIHsKICBmaWxs
OiB0cmFuc3BhcmVudDsKfQoKLnN2Z2JvYiAuZW5kX21hcmtlZF9hcnJvdyB7CiAgbWFya2VyLWVu
ZDogdXJsKCNhcnJvdyk7Cn0KCi5zdmdib2IgLnN0YXJ0X21hcmtlZF9hcnJvdyB7CiAgbWFya2Vy
LXN0YXJ0OiB1cmwoI2Fycm93KTsKfQoKLnN2Z2JvYiAuZW5kX21hcmtlZF9kaWFtb25kIHsKICBt
YXJrZXItZW5kOiB1cmwoI2RpYW1vbmQpOwp9Cgouc3ZnYm9iIC5zdGFydF9tYXJrZWRfZGlhbW9u
ZCB7CiAgbWFya2VyLXN0YXJ0OiB1cmwoI2RpYW1vbmQpOwp9Cgouc3ZnYm9iIC5lbmRfbWFya2Vk
X2NpcmNsZSB7CiAgbWFya2VyLWVuZDogdXJsKCNjaXJjbGUpOwp9Cgouc3ZnYm9iIC5zdGFydF9t
YXJrZWRfY2lyY2xlIHsKICBtYXJrZXItc3RhcnQ6IHVybCgjY2lyY2xlKTsKfQoKLnN2Z2JvYiAu
ZW5kX21hcmtlZF9vcGVuX2NpcmNsZSB7CiAgbWFya2VyLWVuZDogdXJsKCNvcGVuX2NpcmNsZSk7
Cn0KCi5zdmdib2IgLnN0YXJ0X21hcmtlZF9vcGVuX2NpcmNsZSB7CiAgbWFya2VyLXN0YXJ0OiB1
cmwoI29wZW5fY2lyY2xlKTsKfQoKLnN2Z2JvYiAuZW5kX21hcmtlZF9iaWdfb3Blbl9jaXJjbGUg
ewogIG1hcmtlci1lbmQ6IHVybCgjYmlnX29wZW5fY2lyY2xlKTsKfQoKLnN2Z2JvYiAuc3RhcnRf
bWFya2VkX2JpZ19vcGVuX2NpcmNsZSB7CiAgbWFya2VyLXN0YXJ0OiB1cmwoI2JpZ19vcGVuX2Np
cmNsZSk7Cn0KCjwvc3R5bGU+CiAgPGRlZnM+CiAgICA8bWFya2VyIGlkPSJhcnJvdyIgdmlld0Jv
eD0iLTIgLTIgOCA4IiByZWZYPSI0IiByZWZZPSIyIiBtYXJrZXJXaWR0aD0iNyIgbWFya2VySGVp
Z2h0PSI3IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAgIDxwb2x5Z29uIHBvaW50
cz0iMCwwIDAsNCA0LDIgMCwwIj48L3BvbHlnb24+CiAgICA8L21hcmtlcj4KICAgIDxtYXJrZXIg
aWQ9ImRpYW1vbmQiIHZpZXdCb3g9Ii0yIC0yIDggOCIgcmVmWD0iNCIgcmVmWT0iMiIgbWFya2Vy
V2lkdGg9IjciIG1hcmtlckhlaWdodD0iNyIgb3JpZW50PSJhdXRvLXN0YXJ0LXJldmVyc2UiPgog
ICAgICA8cG9seWdvbiBwb2ludHM9IjAsMiAyLDAgNCwyIDIsNCAwLDIiPjwvcG9seWdvbj4KICAg
IDwvbWFya2VyPgogICAgPG1hcmtlciBpZD0iY2lyY2xlIiB2aWV3Qm94PSIwIDAgOCA4IiByZWZY
PSI0IiByZWZZPSI0IiBtYXJrZXJXaWR0aD0iNyIgbWFya2VySGVpZ2h0PSI3IiBvcmllbnQ9ImF1
dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAgIDxjaXJjbGUgY3g9IjQiIGN5PSI0IiByPSIyIiBjbGFz
cz0iZmlsbGVkIj48L2NpcmNsZT4KICAgIDwvbWFya2VyPgogICAgPG1hcmtlciBpZD0ib3Blbl9j
aXJjbGUiIHZpZXdCb3g9IjAgMCA4IDgiIHJlZlg9IjQiIHJlZlk9IjQiIG1hcmtlcldpZHRoPSI3
IiBtYXJrZXJIZWlnaHQ9IjciIG9yaWVudD0iYXV0by1zdGFydC1yZXZlcnNlIj4KICAgICAgPGNp
cmNsZSBjeD0iNCIgY3k9IjQiIHI9IjIiIGNsYXNzPSJiZ19maWxsZWQiPjwvY2lyY2xlPgogICAg
PC9tYXJrZXI+CiAgICA8bWFya2VyIGlkPSJiaWdfb3Blbl9jaXJjbGUiIHZpZXdCb3g9IjAgMCA4
IDgiIHJlZlg9IjQiIHJlZlk9IjQiIG1hcmtlcldpZHRoPSI3IiBtYXJrZXJIZWlnaHQ9IjciIG9y
aWVudD0iYXV0by1zdGFydC1yZXZlcnNlIj4KICAgICAgPGNpcmNsZSBjeD0iNCIgY3k9IjQiIHI9
IjMiIGNsYXNzPSJiZ19maWxsZWQiPjwvY2lyY2xlPgogICAgPC9tYXJrZXI+CiAgPC9kZWZzPgog
IDxyZWN0IGNsYXNzPSJiYWNrZHJvcCIgeD0iMCIgeT0iLTQiIHdpZHRoPSI0NjAuMiIgaGVpZ2h0
PSI4MiI+PC9yZWN0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjEuOTUiIHk9IjEx
LjcwMDAwMSIgPkN5Y2xlOiAgICAgICAxICAgICAgICAgMiAgICAgICAgIDMgICAgICAgICA0ICAg
ICAgICAgNTwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIxLjk1IiB5PSIy
Ny4zMDAwMDEiID5TdGFnZSAxOiBbRmV0Y2ggMV0gW0ZldGNoIDJdIFtGZXRjaCAzXSBbRmV0Y2gg
NF0gW0ZldGNoIDVdPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjEuOTUi
IHk9IjQyLjkiID5TdGFnZSAyOiAgICAgICAgICBbRGVjb2RlIDFdW0RlY29kZSAyXVtEZWNvZGUg
M11bRGVjb2RlIDRdPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjEuOTUi
IHk9IjU4LjUiID5TdGFnZSAzOiAgICAgICAgICAgICAgICAgICAgIFtFeGVjIDEgXSBbRXhlYyAy
IF0gW0V4ZWMgMyBdPC90ZXh0Pgo8L3N2Zz4K
" alt="" />
<figcaption>Figure 9–7: 3-Stage Instruction Pipeline</figcaption>
</figure>

Under ideal conditions, the pipeline achieves an optimal throughput of
**1 instruction per cycle** (Cycles Per Instruction, $\text{CPI} = 1$).

However, conditional jumps introduce **Control Hazards**. When a branch
instruction (e.g., `JZ`) executes in Stage 3, the CPU may discover that
the branch is taken, meaning the pipeline has already fetched and
decoded the wrong subsequent instructions (see Figure 9–8). The pipeline
must be **flushed**, replacing the speculatively fetched instructions
with `NOP` bubbles, which incurs a multi-cycle performance penalty.

<figure id="fig:pipeline-stage">
<img src="data:image/svg+xml;base64,PHN2ZyB2aWV3Qm94PSIyMCAxMCA4MDAgMTAwIiB3aWR0aD0iMTAwJSIgaGVpZ2h0PSIxMDAiIHht
bG5zPSJodHRwOi8vd3d3LnczLm9yZy8yMDAwL3N2ZyI+CiAgPGRlZnM+CiAgICA8c3R5bGU+CiAg
ICAgIDpyb290IHsKICAgICAgICAtLXN2Zy1iZzogdmFyKC0tYmcsICNmZmZmZmYpOwogICAgICAg
IC0tc3ZnLWZnOiB2YXIoLS1mZywgIzAwMDAwMCk7CiAgICAgICAgLS1zdmctdGludC0xLWZhbGxi
YWNrOiAjY2NjY2NjOyAvKiAyMCUgYmxhY2sgKyA4MCUgd2hpdGUgKi8KICAgICAgICAtLXN2Zy10
aW50LTItZmFsbGJhY2s6ICM5OTk5OTk7IC8qIDQwJSBibGFjayArIDYwJSB3aGl0ZSAqLwogICAg
ICAgIC0tc3ZnLXRpbnQtMTogY29sb3ItbWl4KGluIHNyZ2IsIHZhcigtLXN2Zy1mZykgMjAlLCB2
YXIoLS1zdmctYmcpIDgwJSk7CiAgICAgICAgLS1zdmctdGludC0yOiBjb2xvci1taXgoaW4gc3Jn
YiwgdmFyKC0tc3ZnLWZnKSA0MCUsIHZhcigtLXN2Zy1iZykgNjAlKTsKICAgICAgfQogICAgICB0
ZXh0ICAgeyBmb250LWZhbWlseTogIkFyaWFsIE5hcnJvdyIsIEZyZWVTYW5zLCBzYW5zLXNlcmlm
OyB9CiAgICA8L3N0eWxlPgogIDwvZGVmcz4KICA8dGV4dCB4PSIzMCIgeT0iMzAiIGZvbnQtc2l6
ZT0iMTgiIGZvbnQtd2VpZ2h0PSJib2xkIiBmaWxsPSJ2YXIoLS1zdmctZmcpIj5CcmFuY2ggSGF6
YXJkIEZsdXNoIFBlbmFsdHkgKEJyYW5jaCBUYWtlbiBhdCBDeWNsZSAzKTwvdGV4dD4KICA8cmVj
dCB4PSIzMCIgeT0iNDUiIHdpZHRoPSIxNDAiIGhlaWdodD0iMzAiIGZpbGw9InZhcigtLXN2Zy10
aW50LTEsIHZhcigtLXN2Zy10aW50LTEtZmFsbGJhY2spKSIgc3Ryb2tlPSJ2YXIoLS1zdmctZmcp
Ii8+CiAgPHRleHQgeD0iNDAiIHk9IjY1IiBmb250LXNpemU9IjE0IiBmaWxsPSJ2YXIoLS1zdmct
ZmcpIj5DMTogRmV0Y2ggSlo8L3RleHQ+CiAgPHJlY3QgeD0iMTgwIiB5PSI0NSIgd2lkdGg9IjE0
MCIgaGVpZ2h0PSIzMCIgZmlsbD0idmFyKC0tc3ZnLXRpbnQtMSwgdmFyKC0tc3ZnLXRpbnQtMS1m
YWxsYmFjaykpIiBzdHJva2U9InZhcigtLXN2Zy1mZykiLz4KICA8dGV4dCB4PSIxOTAiIHk9IjY1
IiBmb250LXNpemU9IjE0IiBmaWxsPSJ2YXIoLS1zdmctZmcpIj5DMjogRmV0Y2ggSlorMTwvdGV4
dD4KICA8cmVjdCB4PSIzMzAiIHk9IjQ1IiB3aWR0aD0iMTQwIiBoZWlnaHQ9IjMwIiBmaWxsPSJ2
YXIoLS1zdmctdGludC0xLCB2YXIoLS1zdmctdGludC0xLWZhbGxiYWNrKSkiIHN0cm9rZT0idmFy
KC0tc3ZnLWZnKSIvPgogIDx0ZXh0IHg9IjM0MCIgeT0iNjUiIGZvbnQtc2l6ZT0iMTQiIGZpbGw9
InZhcigtLXN2Zy1mZykiPkMzOiBGZXRjaCBKWisyPC90ZXh0PgogIDx0ZXh0IHg9IjMzMCIgeT0i
OTUiIGZvbnQtc2l6ZT0iMTQiIGZpbGw9InZhcigtLXN2Zy1mZykiPuKGkSBCcmFuY2ggVGFrZW4g
4oCUIEZsdXNoIHBpcGVsaW5lPC90ZXh0PgogIDxyZWN0IHg9IjQ4MCIgeT0iNDUiIHdpZHRoPSIx
NDAiIGhlaWdodD0iMzAiIGZpbGw9InZhcigtLXN2Zy10aW50LTEsIHZhcigtLXN2Zy10aW50LTEt
ZmFsbGJhY2spKSIgc3Ryb2tlPSJ2YXIoLS1zdmctZmcpIiBzdHJva2UtZGFzaGFycmF5PSIyLDIi
Lz4KICA8dGV4dCB4PSI0OTAiIHk9IjY1IiBmb250LXNpemU9IjE0IiBmaWxsPSJ2YXIoLS1zdmct
ZmcpIj5bQnViYmxlIE5PUF08L3RleHQ+CiAgPHJlY3QgeD0iNjMwIiB5PSI0NSIgd2lkdGg9IjE2
MCIgaGVpZ2h0PSIzMCIgZmlsbD0idmFyKC0tc3ZnLXRpbnQtMSwgdmFyKC0tc3ZnLXRpbnQtMS1m
YWxsYmFjaykpIiBzdHJva2U9InZhcigtLXN2Zy1mZykiLz4KICA8dGV4dCB4PSI2NDAiIHk9IjY1
IiBmb250LXNpemU9IjE0IiBmaWxsPSJ2YXIoLS1zdmctZmcpIj5DNTogRmV0Y2ggVGFyZ2V0PC90
ZXh0Pgo8L3N2Zz4=
" alt="Figure 9–8: Non-Optimal Pipeline Performance" />
<figcaption>Figure 9–8: Non-Optimal Pipeline Performance</figcaption>
</figure>

**Specification:**

Design a `PipelinedVirtualMachine` that models a 3-stage pipeline:

- `struct PipelineStage { std::uint16_t pc; Instruction instr; bool valid; };`
- Maintain three stage latches: `m_fetch_latch`, `m_decode_latch`, and
  `m_exec_latch`.
- In each cycle, advance all three stages concurrently.
- If an instruction in the execute stage modifies `m_pc` (a taken `JMP`,
  `JZ`, `JNZ`, or `JN`), trigger a **Pipeline Flush**: invalidate the
  fetch and decode latches (injecting 2 cycles of bubble latency) and
  set `m_pc` to the branch target.
- Track total clock cycles and compute effective
  $\text{CPI} = \frac{\text{Cycles}}{\text{Instructions Executed}}$.

``` cpp
struct PipelineRegisters {
    uint16_t fetch_ir{0};
    Instruction decode_instr{Opcode::NOP, AddressingMode::Direct, 0};
    bool flush_requested{false};
};
```

**Verification Tasks:**

- **Task 1 (Linear Execution CPI):** Execute a linear sequence of 10
  `ADD` instructions without jumps. Verify that execution completes in
  exactly 12 cycles (2 cycles to fill the pipeline + 10 execution
  cycles), yielding $\text{CPI} = \frac{12}{10} = 1.2$.
- **Task 2 (Branch Penalty):** Execute a loop that branches back 5
  times. Confirm that each taken branch incurs a 2-cycle flush penalty,
  increasing total cycle count accordingly.
- **Task 3 (Untaken Branch):** Execute a `JZ` instruction when $Z = 0$
  (branch not taken). Verify that no flush occurs and the pipeline
  continues with zero bubble penalty ($\text{CPI} = 1.0$).

#### Exercise 9.12: Vectored Hardware Interrupt Controller (VIC)

**Context and Theory:**

A CPU cannot afford to waste clock cycles continuously polling external
devices to check if new data has arrived. Instead, microprocessors rely
on **Hardware Interrupts**.

An **Interrupt** is an asynchronous signal sent by hardware to the CPU,
demanding immediate attention. When an interrupt is triggered, the CPU
executes the following steps:

1.  It completes the current instruction in the execute stage.
2.  It pushes the current execution context onto the stack:
    $$\text{SP} \leftarrow \text{SP} - 1, \quad \text{RAM}[\text{SP}] \leftarrow \text{PC}$$
    $$\text{SP} \leftarrow \text{SP} - 1, \quad \text{RAM}[\text{SP}] \leftarrow \text{FLAGS}$$
3.  It disables further maskable interrupts to prevent re-entrant
    corruption.
4.  In a **Vectored Interrupt** system, the CPU reads the entry address
    of the corresponding **Interrupt Service Routine (ISR)** from an
    **Interrupt Vector Table (IVT)** located at a fixed memory address
    (e.g., `0x0004` to `0x0007`).
5.  It jumps to the ISR address.
6.  The ISR concludes with an **`IRET` (Interrupt Return)** instruction,
    which pops the saved Flags and Program Counter off the stack,
    seamlessly resuming the interrupted program.

**Specification:**

Implement a Vectored Interrupt Controller for the Mach-16:

- Reserve addresses `0x0004`–`0x0007` as the Interrupt Vector Table
  (IVT).
- Introduce an external method
  `VirtualMachine::raise_interrupt(uint8_t vector_id)` (where
  `vector_id` $\in [0, 3]$).
- Add an interrupt enable flag `m_interrupts_enabled` (toggled by new
  opcodes `CLI` \[disable\] and `SEI` \[enable\]).
- At the boundary between the Execute and Fetch cycles, if an interrupt
  is pending and interrupts are enabled, push `PC` and `FLAGS` to the
  stack, clear `m_interrupts_enabled`, and load `m_pc` from
  `RAM[0x0004 + vector_id]`.
- Implement `IRET`: pops `FLAGS`, then pops `PC`, and re-enables
  interrupts.

``` cpp
void handle_pending_interrupts() {
    if (m_interrupt_pending && m_interrupts_enabled) {
        m_interrupts_enabled = false;
        m_memory[--m_sp] = m_pc;
        m_memory[--m_sp] = pack_flags();
        m_pc = m_memory[0x0004 + (m_pending_vector & 0x03)] & ADDRESS_MASK;
        m_interrupt_pending = false;
    }
}
```

**Verification Tasks:**

- **Task 1 (ISR Execution and Return):** Load a main program that runs
  an infinite loop incrementing `ACC`. Register an ISR at vector `0`
  (`RAM[0x0004] = 0x0050`) that writes `0x00FF` to address `0x0200` and
  executes `IRET`. Trigger interrupt `0`. Verify that RAM `0x0200`
  updates to `0x00FF`, the main loop resumes seamlessly, and the
  Accumulator continues incrementing.
- **Task 2 (Context and Flag Preservation):** Set flags
  ($Z = 1, N = 0$). Trigger an interrupt whose ISR performs operations
  that clear $Z$ and set $N$. Execute `IRET`. Verify that the flags
  register restores exactly to $Z = 1, N = 0$ upon return to the
  interrupted code.
- **Task 3 (Interrupt Masking):** Execute `CLI` to disable interrupts.
  Trigger interrupt `0`. Advance the VM by 10 cycles. Verify that the
  ISR is *not* called while masked, but is serviced immediately once a
  `SEI` instruction re-enables interrupts.